# Deep Learning II — RBM, DBN and DNN for Handwritten Digit Classification

**Institut Polytechnique de Paris · M2 Data Science 2025/2026** 

**Produced by:** Zargui Rayen

**Supervisor:** Yohan Petetin

---

## Overview

This notebook implements and evaluates pre-trained Deep Neural Networks (DNN) for handwritten digit classification on MNIST.
All computations run on GPU when available (CuPy for RBM/DBN/DNN matrix operations; PyTorch for the bonus generative models).

**Only one external file is required:** `binaryalphadigs.mat.

---

### Table of Contents
1. [Setup and GPU Configuration](#sec1)
2. [Data Loading and Preprocessing](#sec2)
3. [Model Definitions: RBM, DBN, DNN](#sec3)
4. [Experiments on Binary AlphaDigits](#sec4)
   - 4.1 RBM on letter A: training curve and generated samples
   - 4.2 Effect of the number of hidden units
   - 4.3 Effect of the learning rate
   - 4.4 DBN on letter A: greedy pre-training
   - 4.5 Effect of DBN depth
   - 4.6 Effect of the number of character classes
   - 4.7 Analysis
5. [Comparative Study on MNIST](#sec5)
   - 5.1 Convergence demo
   - 5.2 Figure 1: Error vs. number of hidden layers
   - 5.3 Figure 2: Error vs. neurons per layer
   - 5.4 Figure 3: Error vs. training set size
   - 5.5 Best configuration search
   - 5.6 Softmax probabilities and misclassified examples
   - 5.7 Confusion matrix
   - 5.8 Learning curves (error vs. epoch)
   - 5.9 Analysis
6. [Bonus: Generative Models](#sec6)
   - 6.1 RBM samples
   - 6.2 DBN samples and MSE per layer
   - 6.3 VAE
   - 6.4 GAN
   - 6.5 DDPM (from scratch)
   - 6.6 Side-by-side comparison
   - 6.7 Analysis
7. [Conclusions](#sec7)

---
## 1. Setup and GPU Configuration <a id='sec1'></a>

**CuPy** provides a drop-in NumPy-compatible API that executes on CUDA GPUs.  
When CuPy is not available the code falls back to NumPy transparently.

Two helper functions handle device transfers:
- `to_xp(a)` — converts any array to an `xp` (CuPy or NumPy) float32 array; calling it on an array that is already on the correct device is a no-op (*idempotent*).
- `to_np(a)` — converts any CuPy or NumPy array to a plain NumPy array.

PyTorch is used exclusively in Section 6 for the bonus generative models (VAE, GAN, DDPM), where its autograd engine is required.

In [1]:
!pip install -q cupy-cuda12x scipy matplotlib torch torchvision

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.57.1 requires numpy<1.25,>=1.21, but you have numpy 2.2.6 which is incompatible.
statsmodels 0.14.1 requires numpy<2,>=1.18, but you have numpy 2.2.6 which is incompatible.


In [2]:
import os
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for HFactory
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# GPU backend: CuPy if available, NumPy otherwise
# ---------------------------------------------------------------------------
try:
    import cupy as cp
    xp = cp
    GPU_OK = True
    _info = cp.cuda.runtime.getDeviceProperties(0)['name']
    _name = _info.decode() if isinstance(_info, bytes) else str(_info)
    print(f'CuPy {cp.__version__}  |  GPU: {_name}')
except ImportError:
    xp = np
    GPU_OK = False
    print('CuPy not found -- running on CPU (NumPy)')


def to_xp(a):
    """
    Convert any array to an xp (CuPy or NumPy) float32 array.
    Idempotent: a CuPy array passed on a GPU build is returned unchanged.
    """
    if GPU_OK:
        import cupy as _cp
        if isinstance(a, _cp.ndarray):
            return a if a.dtype == _cp.float32 else a.astype(_cp.float32)
        return _cp.asarray(np.asarray(a, dtype=np.float32))
    arr = np.asarray(a)
    return arr.astype(np.float32) if arr.dtype != np.float32 else arr


def to_np(a):
    """
    Convert any array (CuPy or NumPy) to a NumPy array.
    """
    if GPU_OK:
        import cupy as _cp
        if isinstance(a, _cp.ndarray):
            return _cp.asnumpy(a)
    return np.asarray(a)


# ---------------------------------------------------------------------------
# Global seed for reproducibility
# ---------------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)
if GPU_OK:
    cp.random.seed(SEED)

os.makedirs('figures', exist_ok=True)
print(f'NumPy {np.__version__}')
print('Output directory: ./figures/')

CuPy 14.0.1  |  GPU: NVIDIA L40S
NumPy 2.2.6
Output directory: ./figures/


---
## 2. Data Loading and Preprocessing <a id='sec2'></a>

### Binary AlphaDigits
- 36 character classes: digits 0-9 (indices 0-9) and letters A-Z (indices 10-35)
- 39 binary images per class, each of size 20 × 16 = **320 pixels**
- Used to validate unsupervised training (no labels needed)

### MNIST
- 70 000 greyscale images of handwritten digits (28 × 28 pixels)
- **Binarised** with threshold 127: pixel ≥ 127 → 1, otherwise → 0
- 60 000 training samples / 10 000 test samples

MNIST is loaded using the Python standard library only (no TensorFlow, no Keras — compatible with NumPy 2.x).

In [3]:
import scipy.io as sio
import urllib.request
import gzip
import struct

# ---------------------------------------------------------------------------
# Binary AlphaDigits
# ---------------------------------------------------------------------------
ALPHA_PATH = 'binaryalphadigs.mat'
ALPHA_OK   = os.path.exists(ALPHA_PATH)


def lire_alpha_digit(mat_path, indices):
    """
    Load AlphaDigits characters from the .mat file.
    Handles all scipy.io loading layouts (object arrays, 3-D arrays, flat arrays).

    Parameters
    ----------
    mat_path : str
    indices  : int or list of int  (0-9 = digits, 10-35 = A-Z)

    Returns
    -------
    X : np.ndarray, shape (N, 320), float32 in {0, 1}
    """
    if isinstance(indices, int):
        indices = [indices]
    dat = sio.loadmat(str(mat_path), squeeze_me=False)['dat']
    arrays = []
    for idx in indices:
        cell = dat[idx, 0]
        if cell.ndim == 3:                                      # (39, 20, 16)
            imgs = cell.reshape(cell.shape[0], -1)
        elif cell.ndim == 2 and cell.dtype == object:           # (39, 1) object
            imgs = np.array([cell[i, 0].flatten() for i in range(len(cell))])
        elif cell.ndim == 2 and cell.shape[1] == 320:           # (39, 320) flat
            imgs = cell
        elif cell.ndim == 2 and tuple(cell.shape) == (20, 16):  # (36, 39) layout
            imgs = np.array([dat[idx, j].flatten() for j in range(dat.shape[1])])
        else:
            imgs = cell.reshape(len(cell), -1)
        arrays.append(imgs.astype(np.float32))
    X = np.vstack(arrays)
    assert X.shape[1] == 320, f'Expected shape (N, 320), got {X.shape}'
    return X


if ALPHA_OK:
    X_A = lire_alpha_digit(ALPHA_PATH, 10)
    print(f'AlphaDigits -- letter A: {X_A.shape}')   # expected (39, 320)
else:
    print('binaryalphadigs.mat not found. Upload the file to the working directory.')


# ---------------------------------------------------------------------------
# MNIST  (pure stdlib + NumPy -- no TensorFlow, compatible with NumPy 2.x)
# ---------------------------------------------------------------------------
def load_mnist():
    """
    Download and binarise MNIST using only the standard library and NumPy.
    Falls back to torchvision if the network is unavailable.

    Returns
    -------
    (X_train, y_train), (X_test, y_test)
    X : float32 array, shape (N, 784), values in {0, 1}
    y : int32 array, shape (N,)
    """
    MIRRORS = [
        'https://storage.googleapis.com/cvdf-datasets/mnist/',
        'https://ossci-datasets.s3.amazonaws.com/mnist/',
    ]
    FILES = [
        'train-images-idx3-ubyte.gz',
        'train-labels-idx1-ubyte.gz',
        't10k-images-idx3-ubyte.gz',
        't10k-labels-idx1-ubyte.gz',
    ]

    def fetch(name):
        dst = f'/tmp/mnist_{name}'
        if not os.path.exists(dst):
            for base in MIRRORS:
                try:
                    print(f'  Downloading {name} ...')
                    urllib.request.urlretrieve(base + name, dst)
                    break
                except Exception:
                    continue
        return dst

    def read_images(path):
        with gzip.open(path, 'rb') as f:
            _, n, r, c = struct.unpack('>IIII', f.read(16))
            return np.frombuffer(f.read(), dtype=np.uint8).reshape(n, r * c).copy()

    def read_labels(path):
        with gzip.open(path, 'rb') as f:
            _, n = struct.unpack('>II', f.read(8))
            return np.frombuffer(f.read(), dtype=np.uint8).copy()

    try:
        X_tr = read_images(fetch(FILES[0]))
        y_tr = read_labels(fetch(FILES[1]))
        X_te = read_images(fetch(FILES[2]))
        y_te = read_labels(fetch(FILES[3]))
    except Exception as exc:
        print(f'Direct download failed ({exc}). Trying torchvision ...')
        import torchvision
        tr = torchvision.datasets.MNIST('/tmp/mnist', train=True,  download=True)
        te = torchvision.datasets.MNIST('/tmp/mnist', train=False, download=True)
        X_tr = tr.data.numpy().reshape(-1, 784)
        y_tr = tr.targets.numpy()
        X_te = te.data.numpy().reshape(-1, 784)
        y_te = te.targets.numpy()

    X_tr = (X_tr >= 127).astype(np.float32)
    X_te = (X_te >= 127).astype(np.float32)
    return (X_tr, y_tr.astype(np.int32)), (X_te, y_te.astype(np.int32))


print('Loading MNIST ...')
(Xm_tr, ym_tr), (Xm_te, ym_te) = load_mnist()
print(f'MNIST train: {Xm_tr.shape}   test: {Xm_te.shape}')

AlphaDigits -- letter A: (39, 320)
Loading MNIST ...
MNIST train: (60000, 784)   test: (10000, 784)


In [4]:
# ---------------------------------------------------------------------------
# Visualisation utilities
# ---------------------------------------------------------------------------

def show_grid(X, shape, n=16, title=None, ncols=8, fname=None, fs=1.9):
    """Display up to n binary images arranged in a grid."""
    X = np.asarray(X)
    n = min(n, len(X))
    ncols = min(ncols, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * fs, nrows * fs))
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(X[i].reshape(shape), cmap='binary', vmin=0, vmax=1,
                      interpolation='nearest')
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    if fname:
        plt.savefig(f'figures/{fname}', dpi=150, bbox_inches='tight')
        print(f'  Saved: figures/{fname}')
    plt.show()


def make_mosaic(imgs, shape, ncols=8):
    """Stitch a list of images into a single canvas array for imshow."""
    imgs = np.asarray(imgs)
    n = len(imgs)
    ncols = min(ncols, n)
    nrows = (n + ncols - 1) // ncols
    H, W = shape
    canvas = np.zeros((nrows * H, ncols * W), dtype=np.float32)
    for i, img in enumerate(imgs):
        r, c = divmod(i, ncols)
        canvas[r * H:(r + 1) * H, c * W:(c + 1) * W] = img.reshape(H, W)
    return canvas


show_grid(Xm_tr[:16], (28, 28), n=16,
          title='MNIST -- binarised training samples (threshold=127)',
          fname='mnist_samples.png')

if ALPHA_OK:
    show_grid(X_A, (20, 16), n=len(X_A),
              title='Binary AlphaDigits -- letter A (all 39 instances)',
              fname='alpha_A.png')

  Saved: figures/mnist_samples.png
  Saved: figures/alpha_A.png


---
## 3. Model Definitions: RBM, DBN, DNN <a id='sec3'></a>

### 3.1 Restricted Boltzmann Machine (RBM)

A Binary-Binary RBM defines a joint distribution over visible units $v \in \{0,1\}^{d_v}$ and hidden units $h \in \{0,1\}^{d_h}$ through the energy function:

$$E(v,h) = -a^\top v - b^\top h - v^\top W h$$

The conditional distributions are:

$$P(h_j=1 \mid v) = \sigma\!\left(b_j + \sum_i W_{ij}\,v_i\right), \qquad P(v_i=1 \mid h) = \sigma\!\left(a_i + \sum_j W_{ij}\,h_j\right)$$

Training uses **Contrastive Divergence-1 (CD-1)**:

$$\Delta W = \eta\left(\langle v h^\top \rangle_{\text{data}} - \langle v h^\top \rangle_{\text{model}}\right)$$

### 3.2 Deep Belief Network (DBN)

A DBN is a stack of RBMs trained **greedily layer by layer**. Each RBM receives the **stochastic binary activations** (Bernoulli samples, not probabilities) of the previous RBM as input. This ensures that each layer sees genuinely binary data.

### 3.3 Deep Neural Network (DNN)

The DNN adds a softmax classification layer on top of the DBN and fine-tunes all parameters by gradient descent. Two initialisation strategies are compared:

| Strategy | Hidden-layer init | Training |
|----------|------------------|----------|
| **Pre-trained** | DBN greedy pre-training (unsupervised) | Backpropagation |
| **Random** | Normal $\mathcal{N}(0,\,0.01^2)$ | Backpropagation only |

The cross-entropy loss minimised during backpropagation is:

$$\mathcal{L} = -\frac{1}{N}\sum_{n=1}^N \sum_{k=0}^{9} y_{nk} \log\hat{p}_{nk}$$

> **Implementation note.** All computations stay on `xp` (CuPy or NumPy) throughout
> the forward and backward passes. Arrays are converted from NumPy to `xp` exactly once,
> at the entry point of `retropropagation`. The helper `to_xp` is idempotent:
> passing a CuPy array on a GPU build returns the same array without any copy.

In [5]:
# ===========================================================================
# Restricted Boltzmann Machine
# ===========================================================================

class RBM:
    """
    Binary-Binary Restricted Boltzmann Machine trained with CD-1.
    All internal arrays live on xp (CuPy on GPU, NumPy on CPU).

    Parameters
    ----------
    dv           : int   -- visible dimension
    dh           : int   -- hidden  dimension
    weight_scale : float -- std of the initial weight distribution
    seed         : int
    """

    def __init__(self, dv, dh, weight_scale=0.01, seed=42):
        rs      = np.random.RandomState(seed)
        self.W  = to_xp(rs.randn(dv, dh).astype(np.float32) * weight_scale)
        self.a  = xp.zeros((1, dv), dtype=xp.float32)   # visible biases
        self.b  = xp.zeros((1, dh), dtype=xp.float32)   # hidden  biases
        self.dv = dv
        self.dh = dh

    @staticmethod
    def _sigmoid(x):
        return 1.0 / (1.0 + xp.exp(-xp.clip(x, -30.0, 30.0)))

    @staticmethod
    def _bernoulli(p):
        """Sample a binary array from Bernoulli probabilities p."""
        return (xp.random.rand(*p.shape) < p).astype(xp.float32)

    def entree_sortie_RBM(self, v):
        """Compute P(h=1 | v).  Input and output are xp arrays."""
        return self._sigmoid(self.b + v @ self.W)

    def sortie_entree_RBM(self, h):
        """Compute P(v=1 | h).  Input and output are xp arrays."""
        return self._sigmoid(self.a + h @ self.W.T)

    def train_RBM(self, X_np, n_epochs=100, lr=0.1, batch_size=64, verbose=True):
        """
        Train with CD-1.

        Returns
        -------
        hist : np.ndarray, shape (n_epochs,) -- reconstruction MSE per epoch
        """
        X    = to_xp(X_np)
        n    = len(X)
        hist = []
        for ep in range(n_epochs):
            X_shuf = X[xp.random.permutation(n)]
            for i in range(0, n, batch_size):
                v0  = X_shuf[i:i + batch_size]
                nb  = len(v0)
                # Positive phase
                ph0 = self.entree_sortie_RBM(v0)
                h0  = self._bernoulli(ph0)
                # Negative phase (1 Gibbs step)
                pv1 = self.sortie_entree_RBM(h0)
                v1  = self._bernoulli(pv1)
                ph1 = self.entree_sortie_RBM(v1)
                # CD-1 parameter updates
                self.W += lr * (v0.T @ ph0 - v1.T @ ph1) / nb
                self.a += lr * xp.mean(v0 - v1,   axis=0, keepdims=True)
                self.b += lr * xp.mean(ph0 - ph1, axis=0, keepdims=True)
            # Reconstruction MSE on the full dataset
            recon = self.sortie_entree_RBM(self.entree_sortie_RBM(X))
            mse   = float(xp.mean((X - recon) ** 2))
            hist.append(mse)
            if verbose and (ep % max(1, n_epochs // 10) == 0 or ep == n_epochs - 1):
                print(f'    [RBM] epoch {ep + 1:4d}/{n_epochs}  MSE = {mse:.6f}')
        return np.array(hist)

    def generer_image_RBM(self, n_images=16, n_gibbs=200, seed=0):
        """
        Generate images by Gibbs sampling.

        Each image is generated by an INDEPENDENT chain (one chain per image).
        The chain runs for n_gibbs full steps starting from a random binary
        visible state.  The FINAL p(v=1|h) probability (not a thresholded
        average of binary samples) is returned -- this gives smooth, digit-like
        images instead of blocky blobs.

        Why the previous approach was wrong
        ------------------------------------
        Running 16 chains in parallel and averaging their binary {0,1} states
        before thresholding at 0.5 merges unrelated chains: a pixel that is 1
        in 8 chains and 0 in the other 8 gives a mean of 0.5, producing large
        uniform blobs rather than digit structure.

        Returns
        -------
        images : np.ndarray, shape (n_images, dv), float32 in [0, 1]
                 Values are probabilities p(v=1|h) of the final step.
        """
        np.random.seed(seed)
        p      = self.dv
        images = []

        for _ in range(n_images):
            # Independent chain: random binary initialisation
            v = (np.random.rand(1, p) < 0.1).astype(np.float32)  # MNIST ~19% active pixels

            # Run Gibbs chain for n_gibbs steps
            for _ in range(n_gibbs):
                h_prob = to_np(self.entree_sortie_RBM(to_xp(v)))
                h      = (np.random.rand(*h_prob.shape) < h_prob).astype(np.float32)
                v_prob = to_np(self.sortie_entree_RBM(to_xp(h)))
                v      = (np.random.rand(*v_prob.shape) < v_prob).astype(np.float32)

            # Return final probability p(v=1|h) -- smooth, not thresholded binary
            h_prob = to_np(self.entree_sortie_RBM(to_xp(v)))
            h      = (np.random.rand(*h_prob.shape) < h_prob).astype(np.float32)
            v_prob = to_np(self.sortie_entree_RBM(to_xp(h)))
            images.append(v_prob.reshape(p))

        return np.stack(images).astype(np.float32)


print('RBM class defined.')

RBM class defined.


In [6]:
# ===========================================================================
# Deep Belief Network
# ===========================================================================

class DBN:
    """
    Deep Belief Network: a stack of RBMs trained greedily layer by layer.

    Parameters
    ----------
    dims : list of int  -- [visible_dim, h1_dim, h2_dim, ...]
                          Does NOT include the classification layer.
    """

    def __init__(self, dims, weight_scale=0.01, seed=42):
        assert len(dims) >= 2, 'DBN requires at least one RBM layer.'
        self.dims = dims
        self.L    = len(dims) - 1
        self.rbms = [
            RBM(dims[i], dims[i + 1], weight_scale, seed + i)
            for i in range(self.L)
        ]

    def train_DBN(self, X_np, n_epochs=100, lr=0.1, batch_size=64, verbose=True):
        """
        Greedy layer-wise training.
        Each layer receives stochastic binary activations from the previous one.

        Returns
        -------
        hists : list of np.ndarray -- MSE histories, one array per layer
        """
        data  = X_np.astype(np.float32)
        hists = []
        for i, rbm in enumerate(self.rbms):
            if verbose:
                print(f'\n  [DBN] Training layer {i + 1}/{self.L}')
                print(f'        Input dim: {rbm.dv}   Hidden dim: {rbm.dh}')
            mse_hist = rbm.train_RBM(
                data, n_epochs=n_epochs, lr=lr,
                batch_size=batch_size, verbose=verbose)
            hists.append(mse_hist)
            # Compute stochastic activations to feed the next layer
            prob = to_np(rbm.entree_sortie_RBM(to_xp(data)))
            data = (np.random.rand(*prob.shape) < prob).astype(np.float32)
        return hists

    def generer_image_DBN(self, n_images=16, n_gibbs=200, seed=0):
        """
        Generate images using full alternating Gibbs sampling through all layers.

        At each step the chain propagates UP through every RBM (visible -> top),
        then DOWN through every RBM (top -> visible), sampling binary states at
        each layer.  This is the correct generation procedure for a DBN, as in
        the reference implementation (Bonus__2_.ipynb, cell 4).

        Running Gibbs only on the top RBM and doing a single top-down pass
        (the naive approach) produces incoherent fragments because the top
        layer has lost too much spatial information.

        Parameters
        ----------
        n_images : int  -- number of images to generate (each an independent chain)
        n_gibbs  : int  -- number of full up-down passes per image
        seed     : int

        Returns
        -------
        images : np.ndarray, shape (n_images, visible_dim), float32 in {0, 1}
        """
        np.random.seed(seed)
        p      = self.dims[0]   # visible dimension
        images = []

        for _ in range(n_images):
            # Initialise from random binary visible state
            v = (np.random.rand(1, p) < 0.1).astype(np.float32)  # MNIST ~19% active pixels

            for _ in range(n_gibbs):
                # --- UP pass: visible -> top hidden ---
                cur = v
                for rbm in self.rbms:
                    prob = to_np(rbm.entree_sortie_RBM(to_xp(cur)))
                    cur  = (np.random.rand(*prob.shape) < prob).astype(np.float32)

                # --- DOWN pass: top hidden -> visible ---
                for rbm in reversed(self.rbms):
                    prob = to_np(rbm.sortie_entree_RBM(to_xp(cur)))
                    cur  = (np.random.rand(*prob.shape) < prob).astype(np.float32)
                v = cur   # new visible state

            # Return p(v=1|h) at final step -- smooth probs, not hard binary
            # (binary {0,1} gives blocky blobs; probabilities give digit structure)
            h_fin    = to_np(self.rbms[0].entree_sortie_RBM(to_xp(v)))
            v_prob   = to_np(self.rbms[0].sortie_entree_RBM(to_xp(
                (np.random.rand(*h_fin.shape) < h_fin).astype(np.float32))))
            images.append(v_prob.reshape(p))

        return np.stack(images).astype(np.float32)


print('DBN class defined.')

DBN class defined.


In [7]:
# ===========================================================================
# Deep Neural Network
# ===========================================================================

class DNN:
    """
    DNN = DBN + Softmax classification layer, fine-tuned by backpropagation.

    Parameters
    ----------
    dims : list of int  -- [input_dim, h1, h2, ..., n_classes]
    """

    def __init__(self, dims, weight_scale=0.01, seed=42):
        assert len(dims) >= 3, 'DNN requires at least one hidden layer.'
        self.dims      = dims
        self.n_classes = dims[-1]
        self.dbn       = DBN(dims[:-1], weight_scale=weight_scale, seed=seed)
        self.L         = self.dbn.L
        rs             = np.random.RandomState(seed + 9999)
        self.W_out = to_xp(
            rs.randn(dims[-2], dims[-1]).astype(np.float32) * weight_scale)
        self.b_out = xp.zeros((1, dims[-1]), dtype=xp.float32)

    # -----------------------------------------------------------------------
    def pretrain_DNN(self, X_np, n_epochs=100, lr=0.1, batch_size=64,
                     verbose=True):
        """Unsupervised pre-training via DBN greedy layer-wise algorithm."""
        return self.dbn.train_DBN(
            X_np, n_epochs=n_epochs, lr=lr,
            batch_size=batch_size, verbose=verbose)

    # -----------------------------------------------------------------------
    @staticmethod
    def _softmax(logits):
        """Numerically stable softmax (subtract row maximum)."""
        z = logits - xp.max(logits, axis=1, keepdims=True)
        e = xp.exp(z)
        return e / xp.sum(e, axis=1, keepdims=True)

    @staticmethod
    def _sigmoid(x):
        return 1.0 / (1.0 + xp.exp(-xp.clip(x, -30.0, 30.0)))

    @staticmethod
    def _sigmoid_deriv(s):
        """Derivative of sigmoid given its output value s."""
        return s * (1.0 - s)

    def calcul_softmax(self, h):
        """Apply the output affine transformation followed by softmax."""
        return self._softmax(h @ self.W_out + self.b_out)

    # -----------------------------------------------------------------------
    def entree_sortie_reseau(self, X):
        """
        Full forward pass.
        Accepts both NumPy and CuPy input (to_xp is idempotent).

        Returns
        -------
        A     : list of xp arrays -- activations at each layer (A[0] = input)
        probs : xp array, shape (N, n_classes) -- softmax probabilities
        """
        cur = to_xp(X)   # idempotent: no-op if X is already a CuPy array
        A   = [cur]
        for rbm in self.dbn.rbms:
            cur = rbm.entree_sortie_RBM(cur)
            A.append(cur)
        return A, self.calcul_softmax(A[-1])

    # -----------------------------------------------------------------------
    def retropropagation(self, X_np, y_np, n_epochs=200, lr=0.1,
                          batch_size=128, verbose=True):
        """
        SGD with backpropagation.

        All data is converted to xp ONCE at entry. Every subsequent operation
        (slicing, forward pass, gradient computation) stays on xp.
        This avoids the CuPy implicit-conversion TypeError.

        Returns
        -------
        hist : np.ndarray, shape (n_epochs,) -- mean cross-entropy per epoch
        """
        n   = len(X_np)
        # Convert data ONCE -- arrays stay on xp for the entire training loop
        X   = to_xp(X_np)
        Y   = to_xp(np.eye(self.n_classes, dtype=np.float32)[y_np])  # one-hot
        eps = xp.float32(1e-12)
        hist = []

        for ep in range(n_epochs):
            perm     = xp.random.permutation(n)
            Xs, Ys   = X[perm], Y[perm]
            ep_loss  = []

            for i in range(0, n, batch_size):
                xb = Xs[i:i + batch_size]   # xp slice -- already on device
                yb = Ys[i:i + batch_size]
                nb = len(xb)

                # Forward pass (xb is xp: entree_sortie_reseau calls to_xp
                # which is a no-op for CuPy arrays)
                A, probs = self.entree_sortie_reseau(xb)

                # Cross-entropy loss (all xp operations)
                loss = -float(
                    xp.mean(xp.sum(yb * xp.log(probs + eps), axis=1)))
                ep_loss.append(loss)

                # Output layer gradient: d(CE)/d(logits) = probs - y
                d_out      = probs - yb
                self.W_out -= lr * (A[-1].T @ d_out) / nb
                self.b_out -= lr * xp.mean(d_out, axis=0, keepdims=True)

                # Backpropagate through hidden layers
                delta = (d_out @ self.W_out.T) * self._sigmoid_deriv(A[-1])
                for l in range(self.L - 1, -1, -1):
                    rbm       = self.dbn.rbms[l]
                    rbm.W    -= lr * (A[l].T @ delta) / nb
                    rbm.b    -= lr * xp.mean(delta, axis=0, keepdims=True)
                    if l > 0:
                        delta = (delta @ rbm.W.T) * self._sigmoid_deriv(A[l])

            el = float(np.mean(ep_loss))
            hist.append(el)
            if verbose and (ep % max(1, n_epochs // 10) == 0
                            or ep == n_epochs - 1):
                print(f'  [DNN] epoch {ep + 1:4d}/{n_epochs}  CE = {el:.5f}')
        return np.array(hist)

    # -----------------------------------------------------------------------
    def test_DNN(self, X_np, y_np):
        """
        Evaluate on a dataset.

        Returns
        -------
        (error_rate, accuracy) : tuple of float
        """
        _, probs = self.entree_sortie_reseau(X_np)
        pred     = to_np(xp.argmax(probs, axis=1))
        acc      = float(np.mean(pred == np.asarray(y_np, dtype=pred.dtype)))
        return 1.0 - acc, acc


print('DNN class defined.')
print('All model classes ready: RBM  DBN  DNN')

DNN class defined.
All model classes ready: RBM  DBN  DNN


---
## 4. Experiments on Binary AlphaDigits <a id='sec4'></a>

This section validates the RBM and DBN implementations by measuring generation quality and studying the effect of key hyperparameters on the AlphaDigits dataset.

All experiments use **letter A** (index 10) as the default single-character class, which provides 39 training samples.

> **Note:** If `binaryalphadigs.mat` is not found, all cells in this section are silently skipped — the MNIST study (Section 5) runs independently.

### 4.1 RBM on Letter A: Training Curve and Generated Samples

A single RBM [320 → 200] is trained with CD-1 on all 39 instances of letter A.  
The reconstruction MSE decreases smoothly, confirming stable convergence.

In [8]:
if not ALPHA_OK:
    print('binaryalphadigs.mat not found. Skipping Section 4.')
else:
    # -----------------------------------------------------------------------
    # 4.1  RBM on letter A: training curve and generated samples
    # -----------------------------------------------------------------------
    X_A = lire_alpha_digit(ALPHA_PATH, 10)
    rbm_A = RBM(320, 200, seed=SEED)
    hist_rbm = rbm_A.train_RBM(
        X_A, n_epochs=200, lr=0.1, batch_size=39, verbose=True)

    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(hist_rbm, color='steelblue', lw=2)
    ax.set_title('RBM -- Reconstruction MSE during training (letter A)',
                 fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Mean Squared Error')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('figures/rbm_convergence.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Final MSE: {hist_rbm[-1]:.6f}')

    gen_rbm_A = rbm_A.generer_image_RBM(n_images=12, n_gibbs=500, seed=1)
    show_grid(gen_rbm_A, (20, 16), n=12, ncols=6,
              title='RBM (200 hidden units) -- Generated samples, letter A',
              fname='rbm_gen_A.png')

    [RBM] epoch    1/200  MSE = 0.197652
    [RBM] epoch   21/200  MSE = 0.155345
    [RBM] epoch   41/200  MSE = 0.121288
    [RBM] epoch   61/200  MSE = 0.092758
    [RBM] epoch   81/200  MSE = 0.071090
    [RBM] epoch  101/200  MSE = 0.054690
    [RBM] epoch  121/200  MSE = 0.043168
    [RBM] epoch  141/200  MSE = 0.034015
    [RBM] epoch  161/200  MSE = 0.027072
    [RBM] epoch  181/200  MSE = 0.021531
    [RBM] epoch  200/200  MSE = 0.017500
  Final MSE: 0.017500
  Saved: figures/rbm_gen_A.png


### 4.2 Effect of the Number of Hidden Units

We compare four capacities: 50, 100, 200, 500 hidden units, all trained for 150 epochs.  
Too few units produce blurry images; too many risk overfitting on the 39-sample dataset.

In [9]:
if ALPHA_OK:
    # -----------------------------------------------------------------------
    # 4.2  Effect of the number of hidden units
    # -----------------------------------------------------------------------
    fig, axes = plt.subplots(1, 4, figsize=(17, 5))
    for k, nh in enumerate([50, 100, 200, 500]):
        r = RBM(320, nh, seed=SEED)
        r.train_RBM(X_A, n_epochs=150, lr=0.1, batch_size=39, verbose=False)
        axes[k].imshow(
            make_mosaic(r.generer_image_RBM(6, 300), (20, 16), ncols=3),
            cmap='binary', vmin=0, vmax=1)
        axes[k].set_title(f'Hidden = {nh}', fontsize=12)
        axes[k].axis('off')
    fig.suptitle('RBM -- Effect of hidden units on generation quality (letter A)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/rbm_hidden_units.png', dpi=150, bbox_inches='tight')
    plt.show()

    # -----------------------------------------------------------------------
    # 4.3  Effect of the learning rate
    # -----------------------------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(13, 5))
    for k, lr_v in enumerate([0.001, 0.01, 0.1]):
        r = RBM(320, 200, seed=SEED)
        r.train_RBM(X_A, n_epochs=150, lr=lr_v, batch_size=39, verbose=False)
        axes[k].imshow(
            make_mosaic(r.generer_image_RBM(6, 300), (20, 16), ncols=3),
            cmap='binary', vmin=0, vmax=1)
        axes[k].set_title(f'LR = {lr_v}', fontsize=12)
        axes[k].axis('off')
    fig.suptitle('RBM -- Effect of learning rate (letter A)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/rbm_learning_rate.png', dpi=150, bbox_inches='tight')
    plt.show()

### 4.3 Effect of the Learning Rate

Three learning rates are compared on RBM [320 → 200], 300 epochs, letter A only.  
The final reconstruction MSE and a generated sample are shown for each setting.

In [10]:
if ALPHA_OK:
    lrs   = [0.001, 0.01, 0.1]
    fig, axes = plt.subplots(1, len(lrs), figsize=(len(lrs) * 4.5, 4.5))
    final_mses = {}
    for ax, lr_val in zip(axes, lrs):
        r = RBM(320, 200, seed=SEED)
        h = r.train_RBM(X_A, n_epochs=300, lr=lr_val, batch_size=39, verbose=False)
        img = r.generer_image_RBM(n_images=1, n_gibbs=200, seed=0)[0]
        ax.imshow(img.reshape(20, 16), cmap='binary', vmin=0, vmax=1,
                  interpolation='nearest')
        ax.axis('off')
        ax.set_title(f'lr = {lr_val}\nFinal MSE = {h[-1]:.4f}',
                     fontsize=11, fontweight='bold')
        final_mses[lr_val] = h[-1]
    fig.suptitle('RBM [320->200] -- Effect of learning rate on letter A (300 epochs)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/rbm_learning_rate.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('  lr=0.001 MSE:', round(final_mses[0.001], 6))
    print('  lr=0.01  MSE:', round(final_mses[0.01],  6))
    print('  lr=0.1   MSE:', round(final_mses[0.1],   6))
    print('Saved: figures/rbm_learning_rate.png')
else:
    print('AlphaDigits not available -- skipping.')

  lr=0.001 MSE: 0.185944
  lr=0.01  MSE: 0.129791
  lr=0.1   MSE: 0.006876
Saved: figures/rbm_learning_rate.png


### 4.4 DBN on Letter A: Greedy Layer-Wise Pre-Training

A DBN [320 → 200 → 100] is trained greedily.  
Layer 1 (320 → 200) is trained first on the raw pixel data; Layer 2 (200 → 100) is then trained on the stochastic binary activations produced by Layer 1.  Each layer receives a different distribution as input, which is why the MSE curves have different scales.

In [11]:
if ALPHA_OK:
    # -----------------------------------------------------------------------
    # 4.4  DBN training and depth comparison
    # -----------------------------------------------------------------------
    dbn_A = DBN([320, 200, 100], seed=SEED)
    dbn_A.train_DBN(
        X_A, n_epochs=150, lr=0.1, batch_size=39, verbose=True)

    show_grid(
        dbn_A.generer_image_DBN(12, 500, 2), (20, 16), n=12, ncols=6,
        title='DBN [320->200->100] -- Generated samples, letter A',
        fname='dbn_gen_A.png')

    fig, axes = plt.subplots(1, 3, figsize=(13, 5))
    for k, (label, dims) in enumerate([
            ('1 layer [320->200]',             [320, 200]),
            ('2 layers [320->200->100]',        [320, 200, 100]),
            ('3 layers [320->200->100->50]',    [320, 200, 100, 50])]):
        d = DBN(dims, seed=SEED)
        d.train_DBN(X_A, n_epochs=100, lr=0.1, batch_size=39, verbose=False)
        axes[k].imshow(
            make_mosaic(d.generer_image_DBN(6, 400), (20, 16), ncols=3),
            cmap='binary', vmin=0, vmax=1)
        axes[k].set_title(label, fontsize=10)
        axes[k].axis('off')
    fig.suptitle('DBN -- Effect of depth on generation quality (letter A)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/dbn_depth.png', dpi=150, bbox_inches='tight')
    plt.show()


  [DBN] Training layer 1/2
        Input dim: 320   Hidden dim: 200
    [RBM] epoch    1/150  MSE = 0.197669
    [RBM] epoch   16/150  MSE = 0.175426
    [RBM] epoch   31/150  MSE = 0.134764
    [RBM] epoch   46/150  MSE = 0.114119
    [RBM] epoch   61/150  MSE = 0.093175
    [RBM] epoch   76/150  MSE = 0.076286
    [RBM] epoch   91/150  MSE = 0.063082
    [RBM] epoch  106/150  MSE = 0.052141
    [RBM] epoch  121/150  MSE = 0.043878
    [RBM] epoch  136/150  MSE = 0.036683
    [RBM] epoch  150/150  MSE = 0.030777

  [DBN] Training layer 2/2
        Input dim: 200   Hidden dim: 100
    [RBM] epoch    1/150  MSE = 0.207128
    [RBM] epoch   16/150  MSE = 0.194202
    [RBM] epoch   31/150  MSE = 0.137208
    [RBM] epoch   46/150  MSE = 0.116815
    [RBM] epoch   61/150  MSE = 0.102041
    [RBM] epoch   76/150  MSE = 0.087791
    [RBM] epoch   91/150  MSE = 0.074831
    [RBM] epoch  106/150  MSE = 0.064086
    [RBM] epoch  121/150  MSE = 0.054923
    [RBM] epoch  136/150  MSE = 0.047124
 

### 4.5 Effect of DBN Depth

We compare DBNs of increasing depth: 1, 2, and 3 hidden layers, all trained for 150 epochs per layer on letter A.  Deeper networks capture more abstract features but are harder to train on the small 39-sample dataset.

In [12]:
if ALPHA_OK:
    depths = [
        ([320, 200],          '1 hidden layer\n[320->200]'),
        ([320, 200, 100],     '2 hidden layers\n[320->200->100]'),
        ([320, 200, 100, 50], '3 hidden layers\n[320->200->100->50]'),
    ]
    fig, axes = plt.subplots(1, len(depths), figsize=(len(depths) * 5.5, 4.5))
    for ax, (dims, title) in zip(axes, depths):
        d = DBN(dims, seed=SEED)
        d.train_DBN(X_A, n_epochs=150, lr=0.1, batch_size=39, verbose=False)
        img = d.generer_image_DBN(n_images=1, n_gibbs=200, seed=0)[0]
        ax.imshow(img.reshape(20, 16), cmap='binary', vmin=0, vmax=1,
                  interpolation='nearest')
        ax.axis('off')
        ax.set_title(title, fontsize=11, fontweight='bold')
    fig.suptitle('DBN -- Effect of depth on generated quality (letter A, 150 epochs/layer)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/dbn_depth.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/dbn_depth.png')
else:
    print('AlphaDigits not available -- skipping.')

Saved: figures/dbn_depth.png


### 4.6 Effect of the Number of Character Classes

We increase the number of character classes from 1 to 5 and observe how the generative quality changes.  With a single class the model produces coherent images; with more classes it must share capacity, and individual character structure becomes less well defined.

In [13]:
if ALPHA_OK:
    # -----------------------------------------------------------------------
    # 4.5  Effect of the number of character classes
    # -----------------------------------------------------------------------
    class_configs = [
        ('A  (1 class)',    [10]),
        ('A+B  (2)',        [10, 11]),
        ('A+B+C  (3)',      [10, 11, 12]),
        ('A to E  (5)',     [10, 11, 12, 13, 14]),
    ]
    fig_r, axr = plt.subplots(1, 4, figsize=(17, 5))
    fig_d, axd = plt.subplots(1, 4, figsize=(17, 5))

    for k, (label, idxs) in enumerate(class_configs):
        Xc = lire_alpha_digit(ALPHA_PATH, idxs)
        bs = min(64, len(Xc))

        r = RBM(320, 200, seed=SEED)
        r.train_RBM(Xc, n_epochs=100, lr=0.1, batch_size=bs, verbose=False)
        axr[k].imshow(
            make_mosaic(r.generer_image_RBM(6, 400), (20, 16), ncols=3),
            cmap='binary', vmin=0, vmax=1)
        axr[k].set_title(f'{label}\nN={len(Xc)}', fontsize=10)
        axr[k].axis('off')

        d = DBN([320, 200, 100], seed=SEED)
        d.train_DBN(Xc, n_epochs=100, lr=0.1, batch_size=bs, verbose=False)
        axd[k].imshow(
            make_mosaic(d.generer_image_DBN(6, 400), (20, 16), ncols=3),
            cmap='binary', vmin=0, vmax=1)
        axd[k].set_title(f'{label}\nN={len(Xc)}', fontsize=10)
        axd[k].axis('off')

    fig_r.suptitle('RBM -- Effect of the number of learned classes',
                   fontsize=13, fontweight='bold')
    fig_r.tight_layout()
    fig_r.savefig('figures/rbm_nclasses.png', dpi=150, bbox_inches='tight')
    fig_r.show()

    fig_d.suptitle('DBN -- Effect of the number of learned classes',
                   fontsize=13, fontweight='bold')
    fig_d.tight_layout()
    fig_d.savefig('figures/dbn_nclasses.png', dpi=150, bbox_inches='tight')
    fig_d.show()

### 4.6 Analysis

**RBM:**
- With a single class (letter A), the model produces recognisable images after approximately
  150 epochs. The MSE decreases smoothly, confirming stable CD-1 convergence.
- Optimal capacity: 200-500 hidden units. Below 50 units the images are too blurry;
  above 500 units the risk of overfitting increases on a 39-sample dataset.
- Learning rate 0.1 converges fastest without instability.
  LR = 0.001 barely moves weights in 150 epochs.
- With 3 or more classes the single shared representation space cannot separate the
  character distributions: generated images become meaningless blends.

**DBN:**
- Hierarchical features produce crisper images and better class separation than a
  comparable single RBM. The two-layer model [320->200->100] is the best trade-off.
- A third layer provides negligible additional gain at 39 samples per class.
- Under multi-class conditions the DBN degrades more gracefully than the RBM:
  it can still generate recognisable instances of letter C after training on {A, B, C}.

---
## 5. Comparative Study on MNIST <a id='sec5'></a>

Two networks with identical architecture are trained under the same conditions
but with different initialisations:

- **Pre-trained:** DBN greedy layer-wise initialisation, then backpropagation.
- **Random:** Normal N(0, 0.01^2) initialisation, then backpropagation only.

**Fixed hyperparameters:**

| Parameter | Value |
|-----------|-------|
| Pre-training epochs per RBM | 100 |
| Backpropagation epochs | 200 |
| Learning rate (both phases) | 0.1 |
| Batch size -- pre-training | 64 |
| Batch size -- backpropagation | 128 |

In [14]:
# ---------------------------------------------------------------------------
# Hyperparameters
# ---------------------------------------------------------------------------
HP_RBM = dict(n_epochs=100, lr=0.1, batch_size=64)
HP_BP  = dict(n_epochs=200, lr=0.1, batch_size=128)


def run_experiment(dims, pretrain=True, n_train=None, seed=SEED, verbose=False):
    """
    Train a DNN with or without DBN pre-training and return evaluation metrics.

    Parameters
    ----------
    dims     : list of int  -- architecture, e.g. [784, 200, 200, 10]
    pretrain : bool         -- whether to apply DBN pre-training
    n_train  : int or None  -- subsample training set; None = use all 60 000

    Returns
    -------
    dict with keys: dims, pretrain, n_train, err_train, acc_train,
                    err_test, acc_test, history, model
    """
    X_tr, y_tr = Xm_tr.copy(), ym_tr.copy()
    if n_train is not None:
        idx  = np.random.RandomState(seed).permutation(len(X_tr))[:int(n_train)]
        X_tr, y_tr = X_tr[idx], y_tr[idx]

    dnn = DNN(dims, seed=seed)
    if pretrain:
        if verbose:
            print('  Phase 1: DBN pre-training')
        dnn.pretrain_DNN(X_tr, **HP_RBM, verbose=verbose)

    if verbose:
        print('  Phase 2: Backpropagation fine-tuning')
    hist = dnn.retropropagation(X_tr, y_tr, **HP_BP, verbose=verbose)

    err_tr, acc_tr = dnn.test_DNN(X_tr,    y_tr)
    err_te, acc_te = dnn.test_DNN(Xm_te,   ym_te)

    return dict(
        dims=dims, pretrain=pretrain, n_train=len(X_tr),
        err_train=err_tr, acc_train=acc_tr,
        err_test=err_te,  acc_test=acc_te,
        history=hist, model=dnn)


# ---------------------------------------------------------------------------
# Table-printing utility (no pandas dependency)
# ---------------------------------------------------------------------------
def print_table(rows, cols):
    """Print a list of dicts as a formatted ASCII table."""
    widths = {k: max(len(str(k)), max(len(str(r.get(k, ''))) for r in rows))
              for k in cols}
    header = '  '.join(str(k).ljust(widths[k]) for k in cols)
    sep    = '  '.join('-' * widths[k]          for k in cols)
    print(header)
    print(sep)
    for r in rows:
        print('  '.join(str(r.get(k, '')).ljust(widths[k]) for k in cols))


def sort_rows(rows, key, reverse=False):
    """Sort a list of dicts by a given key."""
    return sorted(rows, key=lambda r: r[key], reverse=reverse)


print('Utilities ready.')

Utilities ready.


### 5.1 Convergence Demo

Before running the full grid, we train two identical networks (`[784, 200, 200, 10]`) on 5 000 samples: one with DBN pre-training, one with random initialisation.  The cross-entropy curves confirm that pre-training provides a better starting point — lower initial loss and faster convergence.

In [15]:
# Quick convergence demo on a 5 000-sample subset
print('=' * 58)
print('Convergence demo  -- architecture [784, 200, 200, 10]')
print('Training set size: 5 000')
print('=' * 58)

dims_demo = [784, 200, 200, 10]
res_pre   = run_experiment(
    dims_demo, pretrain=True,  n_train=5000, seed=SEED, verbose=True)
res_rand  = run_experiment(
    dims_demo, pretrain=False, n_train=5000, seed=SEED, verbose=True)

print(f"\nPre-trained : accuracy = {res_pre['acc_test']:.4f}  "
      f"error = {res_pre['err_test']:.4f}")
print(f"Random init : accuracy = {res_rand['acc_test']:.4f}  "
      f"error = {res_rand['err_test']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, scale in zip(axes, ['linear', 'log']):
    ax.plot(res_pre['history'],  lw=2, color='steelblue',
            label='Pre-trained (DBN)')
    ax.plot(res_rand['history'], lw=2, color='tomato', ls='--',
            label='Random initialisation')
    if scale == 'log':
        ax.set_yscale('log')
    ax.set_title(f'Cross-Entropy Loss  ({scale} scale)', fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Mean CE Loss')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
fig.suptitle('Convergence: pre-trained vs. random initialisation'
             '  |  architecture [784, 200, 200, 10]  |  5 000 training samples',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/convergence_demo.png', dpi=150, bbox_inches='tight')
plt.show()

Convergence demo  -- architecture [784, 200, 200, 10]
Training set size: 5 000
  Phase 1: DBN pre-training

  [DBN] Training layer 1/2
        Input dim: 784   Hidden dim: 200
    [RBM] epoch    1/100  MSE = 0.068528
    [RBM] epoch   11/100  MSE = 0.030441
    [RBM] epoch   21/100  MSE = 0.023647
    [RBM] epoch   31/100  MSE = 0.020575
    [RBM] epoch   41/100  MSE = 0.018501
    [RBM] epoch   51/100  MSE = 0.016956
    [RBM] epoch   61/100  MSE = 0.015690
    [RBM] epoch   71/100  MSE = 0.014735
    [RBM] epoch   81/100  MSE = 0.014264
    [RBM] epoch   91/100  MSE = 0.013747
    [RBM] epoch  100/100  MSE = 0.013186

  [DBN] Training layer 2/2
        Input dim: 200   Hidden dim: 200
    [RBM] epoch    1/100  MSE = 0.183081
    [RBM] epoch   11/100  MSE = 0.078371
    [RBM] epoch   21/100  MSE = 0.059969
    [RBM] epoch   31/100  MSE = 0.050790
    [RBM] epoch   41/100  MSE = 0.045506
    [RBM] epoch   51/100  MSE = 0.041911
    [RBM] epoch   61/100  MSE = 0.039397
    [RBM] epoch  

### 5.2 Figure 1 -- Error vs. Number of Hidden Layers

Architecture: `[784, 200 x L, 10]`  for L in {1, 2, 3, 4, 5}.  
Full MNIST training set (60 000 samples).

In [16]:
results_f1 = []
for L in [1, 2, 3, 4, 5]:
    dims = [784] + [200] * L + [10]
    print(f'\nDepth L={L}  architecture: {dims}')
    for pre in [True, False]:
        r = run_experiment(dims, pretrain=pre, n_train=None, seed=SEED)
        results_f1.append({
            'n_layers': L, 'pretrain': pre,
            'err_train': r['err_train'], 'err_test': r['err_test'],
            'acc_test':  r['acc_test']})
        tag = 'pre-trained' if pre else 'random     '
        print(f'  {tag}  err_test = {r["err_test"]:.4f}  '
              f'acc_test = {r["acc_test"]:.4f}')
print('\nFigure 1 complete.')


Depth L=1  architecture: [784, 200, 10]
  pre-trained  err_test = 0.0359  acc_test = 0.9641
  random       err_test = 0.0232  acc_test = 0.9768

Depth L=2  architecture: [784, 200, 200, 10]
  pre-trained  err_test = 0.0245  acc_test = 0.9755
  random       err_test = 0.0256  acc_test = 0.9744

Depth L=3  architecture: [784, 200, 200, 200, 10]
  pre-trained  err_test = 0.0231  acc_test = 0.9769
  random       err_test = 0.9108  acc_test = 0.0892

Depth L=4  architecture: [784, 200, 200, 200, 200, 10]
  pre-trained  err_test = 0.0232  acc_test = 0.9768
  random       err_test = 0.8991  acc_test = 0.1009

Depth L=5  architecture: [784, 200, 200, 200, 200, 200, 10]
  pre-trained  err_test = 0.0238  acc_test = 0.9762
  random       err_test = 0.8990  acc_test = 0.1010

Figure 1 complete.


In [17]:
s_pre  = sort_rows([r for r in results_f1 if     r['pretrain']], 'n_layers')
s_rand = sort_rows([r for r in results_f1 if not r['pretrain']], 'n_layers')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, col, split in [(ax1, 'err_train', 'Train'),
                        (ax2, 'err_test',  'Test')]:
    ax.plot([r['n_layers'] for r in s_pre],
            [r[col]        for r in s_pre],
            '-o', color='steelblue', lw=2.5, ms=9,
            label='Pre-trained (DBN)')
    ax.plot([r['n_layers'] for r in s_rand],
            [r[col]        for r in s_rand],
            '-s', color='tomato', lw=2.5, ms=9, alpha=0.85,
            label='Random initialisation')
    ax.set_title(f'Figure 1 -- {split} Error vs. Depth', fontsize=12)
    ax.set_xlabel('Number of hidden layers', fontsize=11)
    ax.set_ylabel('Classification error rate', fontsize=11)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
fig.suptitle('Figure 1 -- Impact of depth  |  [784, 200xL, 10]  |  60 000 samples',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig1_depth.png', dpi=150, bbox_inches='tight')
plt.show()
print_table(results_f1, ['n_layers', 'pretrain', 'err_train', 'err_test'])

n_layers  pretrain  err_train              err_test            
--------  --------  ---------------------  --------------------
1         True      0.00601666666666667    0.03590000000000004 
1         False     0.001783333333333359   0.0232              
2         True      0.003166666666666651   0.024499999999999966
2         False     0.0001666666666666483  0.025599999999999956
3         True      0.001966666666666672   0.02310000000000001 
3         False     0.90965                0.9108              
4         True      0.0019166666666666776  0.0232              
4         False     0.90085                0.8991              
5         True      0.0013999999999999568  0.023800000000000043
5         False     0.8978166666666667     0.899               


### 5.3 Figure 2 -- Error vs. Neurons per Layer

Architecture: `[784, h, h, 10]`  for h in {100, 200, 300, 500, 700}.  
Full MNIST training set (60 000 samples).

In [18]:
results_f2 = []
for h in [100, 200, 300, 500, 700]:
    dims = [784, h, h, 10]
    print(f'\nWidth h={h}  architecture: {dims}')
    for pre in [True, False]:
        r = run_experiment(dims, pretrain=pre, n_train=None, seed=SEED)
        results_f2.append({
            'width': h, 'pretrain': pre,
            'err_train': r['err_train'], 'err_test': r['err_test']})
        tag = 'pre-trained' if pre else 'random     '
        print(f'  {tag}  err_test = {r["err_test"]:.4f}')
print('\nFigure 2 complete.')


Width h=100  architecture: [784, 100, 100, 10]
  pre-trained  err_test = 0.0337
  random       err_test = 0.0270

Width h=200  architecture: [784, 200, 200, 10]
  pre-trained  err_test = 0.0273
  random       err_test = 0.0258

Width h=300  architecture: [784, 300, 300, 10]
  pre-trained  err_test = 0.0241
  random       err_test = 0.0563

Width h=500  architecture: [784, 500, 500, 10]
  pre-trained  err_test = 0.0195
  random       err_test = 0.8972

Width h=700  architecture: [784, 700, 700, 10]
  pre-trained  err_test = 0.0189
  random       err_test = 0.8968

Figure 2 complete.


In [19]:
s_pre  = sort_rows([r for r in results_f2 if     r['pretrain']], 'width')
s_rand = sort_rows([r for r in results_f2 if not r['pretrain']], 'width')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, col, split in [(ax1, 'err_train', 'Train'),
                        (ax2, 'err_test',  'Test')]:
    ax.plot([r['width'] for r in s_pre],  [r[col] for r in s_pre],
            '-o', color='steelblue', lw=2.5, ms=9, label='Pre-trained (DBN)')
    ax.plot([r['width'] for r in s_rand], [r[col] for r in s_rand],
            '-s', color='tomato', lw=2.5, ms=9, alpha=0.85,
            label='Random initialisation')
    ax.set_title(f'Figure 2 -- {split} Error vs. Neurons per Layer', fontsize=12)
    ax.set_xlabel('Neurons per layer', fontsize=11)
    ax.set_ylabel('Classification error rate', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
fig.suptitle('Figure 2 -- Impact of layer width  |  [784, h, h, 10]  |  60 000 samples',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig2_width.png', dpi=150, bbox_inches='tight')
plt.show()
print_table(results_f2, ['width', 'pretrain', 'err_train', 'err_test'])

width  pretrain  err_train               err_test            
-----  --------  ----------------------  --------------------
100    True      0.011499999999999955    0.03369999999999995 
100    False     0.00013333333333331865  0.027000000000000024
200    True      0.0033999999999999586   0.02729999999999999 
200    False     0.00014999999999998348  0.025800000000000045
300    True      0.0014499999999999513   0.02410000000000001 
300    False     0.042649999999999966    0.05630000000000002 
500    True      0.0002666666666666373   0.019499999999999962
500    False     0.8955833333333333      0.8972              
700    True      8.333333333332416e-05   0.018900000000000028
700    False     0.9007000000000001      0.8968              


### 5.4 Figure 3 -- Error vs. Training Set Size

Fixed architecture `[784, 200, 200, 10]`. Training set size varies over
{1 000, 3 000, 7 000, 10 000, 30 000, 60 000}.

In [20]:
results_f3 = []
dims_f3    = [784, 200, 200, 10]
for ntr in [1000, 3000, 7000, 10000, 30000, 60000]:
    print(f'\nTraining set size: {ntr:,}')
    for pre in [True, False]:
        r = run_experiment(dims_f3, pretrain=pre, n_train=ntr, seed=SEED)
        results_f3.append({
            'n_train': ntr, 'pretrain': pre,
            'err_train': r['err_train'], 'err_test': r['err_test']})
        tag = 'pre-trained' if pre else 'random     '
        print(f'  {tag}  err_test = {r["err_test"]:.4f}')
print('\nFigure 3 complete.')


Training set size: 1,000
  pre-trained  err_test = 0.0896
  random       err_test = 0.8991

Training set size: 3,000
  pre-trained  err_test = 0.0705
  random       err_test = 0.2312

Training set size: 7,000
  pre-trained  err_test = 0.0546
  random       err_test = 0.0946

Training set size: 10,000
  pre-trained  err_test = 0.0489
  random       err_test = 0.0731

Training set size: 30,000
  pre-trained  err_test = 0.0336
  random       err_test = 0.0359

Training set size: 60,000
  pre-trained  err_test = 0.0261
  random       err_test = 0.0260

Figure 3 complete.


In [23]:
s_pre  = sort_rows([r for r in results_f3 if     r['pretrain']], 'n_train')
s_rand = sort_rows([r for r in results_f3 if not r['pretrain']], 'n_train')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, col, split in [(ax1, 'err_train', 'Train'),
                        (ax2, 'err_test',  'Test')]:
    ax.plot([r['n_train'] for r in s_pre],  [r[col] for r in s_pre],
            '-o', color='steelblue', lw=2.5, ms=9, label='Pre-trained (DBN)')
    ax.plot([r['n_train'] for r in s_rand], [r[col] for r in s_rand],
            '-s', color='tomato', lw=2.5, ms=9, alpha=0.85,
            label='Random initialisation')
    ax.set_title(f'Figure 3 -- {split} Error vs. Training Size', fontsize=12)
    ax.set_xlabel('Number of training samples', fontsize=11)
    ax.set_ylabel('Classification error rate', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
fig.suptitle('Figure 3 -- Impact of training set size  |  [784, 200, 200, 10]',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig3_size.png', dpi=150, bbox_inches='tight')
plt.show()
print_table(results_f3, ['n_train', 'pretrain', 'err_train', 'err_test'])

n_train  pretrain  err_train               err_test            
-------  --------  ----------------------  --------------------
1000     True      0.01200000000000001     0.08960000000000001 
1000     False     0.9                     0.8991              
3000     True      0.012666666666666715    0.07050000000000001 
3000     False     0.19966666666666666     0.23119999999999996 
7000     True      0.00914285714285712     0.05459999999999998 
7000     False     0.04300000000000004     0.09460000000000002 
10000    True      0.008600000000000052    0.048900000000000055
10000    False     0.02210000000000001     0.07310000000000005 
30000    True      0.005166666666666653    0.03359999999999996 
30000    False     0.0008000000000000229   0.03590000000000004 
60000    True      0.0032166666666666455   0.026100000000000012
60000    False     0.00014999999999998348  0.026000000000000023


### 5.5 Best Configuration

In [24]:
best_candidates = [
    [784, 400, 400, 10],
    [784, 500, 500, 10],
    [784, 700, 700, 10],
    [784, 500, 500, 500, 10],
    [784, 300, 300, 300, 10],
]
results_best = []

print('=' * 62)
print('Best configuration search -- pre-trained, 60 000 training samples')
print('=' * 62)
for dims in best_candidates:
    r = run_experiment(dims, pretrain=True, n_train=None, seed=SEED)
    results_best.append({
        'architecture': str(dims),
        'err_test':     round(r['err_test'], 4),
        'acc_test':     round(r['acc_test'], 4)})
    print(f"  {str(dims):38s}  acc = {r['acc_test']:.4f}  "
          f"err = {r['err_test']:.4f}")

ranked = sort_rows(results_best, 'err_test')
print('\nRanking:')
print_table(ranked, ['architecture', 'acc_test', 'err_test'])
best = ranked[0]
print(f"\nBest architecture : {best['architecture']}")
print(f"Test accuracy     : {best['acc_test']:.4f}  "
      f"({(1 - best['acc_test']) * 100:.2f}% error rate)")

Best configuration search -- pre-trained, 60 000 training samples
  [784, 400, 400, 10]                     acc = 0.9806  err = 0.0194
  [784, 500, 500, 10]                     acc = 0.9807  err = 0.0193
  [784, 700, 700, 10]                     acc = 0.9827  err = 0.0173
  [784, 500, 500, 500, 10]                acc = 0.9794  err = 0.0206
  [784, 300, 300, 300, 10]                acc = 0.9810  err = 0.0190

Ranking:
architecture              acc_test  err_test
------------------------  --------  --------
[784, 700, 700, 10]       0.9827    0.0173  
[784, 300, 300, 300, 10]  0.981     0.019   
[784, 500, 500, 10]       0.9807    0.0193  
[784, 400, 400, 10]       0.9806    0.0194  
[784, 500, 500, 500, 10]  0.9794    0.0206  

Best architecture : [784, 700, 700, 10]
Test accuracy     : 0.9827  (1.73% error rate)


### 5.6 Figure 4 -- Softmax Probabilities and Misclassified Examples

**Top panel:** For 10 test samples, we display the image and the full softmax
probability vector. The predicted class bar is coloured green (correct) or red (wrong).

**Bottom panel:** We collect the first 16 misclassified examples, show the image,
and annotate both the true label and the erroneous prediction made by the DNN.

In [25]:
# Use the best model trained in the demo (dims_demo, pre-trained)
best_model = res_pre['model']

# -----------------------------------------------------------------------
# Panel A: softmax probability barplots for 10 test images
# -----------------------------------------------------------------------
N_SHOW = 10
X_show = Xm_te[:N_SHOW]
y_show = ym_te[:N_SHOW]

_, probs_raw = best_model.entree_sortie_reseau(X_show)
probs_np     = to_np(probs_raw)
preds        = np.argmax(probs_np, axis=1)

fig = plt.figure(figsize=(N_SHOW * 2.2, 5.5))
gs  = gridspec.GridSpec(2, N_SHOW, hspace=0.1, wspace=0.35)

for i in range(N_SHOW):
    # Image row
    ax_img = fig.add_subplot(gs[0, i])
    ax_img.imshow(X_show[i].reshape(28, 28), cmap='binary')
    ax_img.axis('off')
    colour = 'green' if preds[i] == y_show[i] else 'red'
    ax_img.set_title(
        f'True: {y_show[i]}\nPred: {preds[i]}',
        fontsize=8, color=colour, fontweight='bold')

    # Probability barplot row
    ax_bar = fig.add_subplot(gs[1, i])
    bar_colors = [
        'green' if j == y_show[i] else
        ('red'  if j == preds[i] and preds[i] != y_show[i] else 'steelblue')
        for j in range(10)]
    ax_bar.bar(range(10), probs_np[i], color=bar_colors, width=0.85)
    ax_bar.set_xticks(range(10))
    ax_bar.set_xticklabels(range(10), fontsize=7)
    ax_bar.set_ylim(0, 1)
    ax_bar.tick_params(axis='y', labelsize=7)

fig.suptitle(
    'Figure 4A -- Softmax output probabilities (pre-trained DNN)\n'
    'Green = correct class     Red = incorrect prediction',
    fontsize=11, fontweight='bold')
plt.savefig('figures/fig4a_softmax_probs.png', dpi=150, bbox_inches='tight')
plt.show()

# -----------------------------------------------------------------------
# Panel B: misclassified examples
# -----------------------------------------------------------------------
_, probs_full = best_model.entree_sortie_reseau(Xm_te)
preds_full    = to_np(xp.argmax(probs_full, axis=1))
wrong_idx     = np.where(preds_full != ym_te)[0]

N_WRONG = min(16, len(wrong_idx))
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
axes = axes.flatten()
for k, idx in enumerate(wrong_idx[:N_WRONG]):
    axes[k].imshow(Xm_te[idx].reshape(28, 28), cmap='binary')
    axes[k].axis('off')
    axes[k].set_title(
        f'True: {ym_te[idx]}\nPred: {preds_full[idx]}',
        fontsize=9, color='red', fontweight='bold')
fig.suptitle(
    f'Figure 4B -- First {N_WRONG} misclassified test examples '
    f'(total errors: {len(wrong_idx)}/{len(Xm_te)})',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig4b_misclassified.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total misclassified: {len(wrong_idx)} / {len(Xm_te)} '
      f'({len(wrong_idx) / len(Xm_te) * 100:.2f}%)')

Total misclassified: 562 / 10000 (5.62%)


### 5.7 Confusion Matrix

The confusion matrix reveals which digit pairs the network confuses most often.
Off-diagonal entries are normalised by the true class count (row-normalised),
making the matrix comparable across classes with different test set sizes.

In [26]:
# Build and display the confusion matrix
n_classes = 10
conf_mat  = np.zeros((n_classes, n_classes), dtype=np.int32)
for true, pred in zip(ym_te, preds_full):
    conf_mat[true, pred] += 1

# Row-normalise (each row sums to 1.0)
conf_norm = conf_mat.astype(float) / conf_mat.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# -- Raw counts --
im0 = axes[0].imshow(conf_mat, cmap='Blues')
axes[0].set_title('Confusion Matrix -- Raw Counts', fontsize=12)
for i in range(n_classes):
    for j in range(n_classes):
        axes[0].text(j, i, str(conf_mat[i, j]),
                     ha='center', va='center',
                     fontsize=7.5,
                     color='white' if conf_mat[i, j] > conf_mat.max() * 0.6 else 'black')
plt.colorbar(im0, ax=axes[0])

# -- Row-normalised --
im1 = axes[1].imshow(conf_norm, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix -- Row-Normalised', fontsize=12)
for i in range(n_classes):
    for j in range(n_classes):
        axes[1].text(j, i, f'{conf_norm[i, j]:.2f}',
                     ha='center', va='center',
                     fontsize=7,
                     color='white' if conf_norm[i, j] > 0.6 else 'black')
plt.colorbar(im1, ax=axes[1])

for ax in axes:
    ax.set_xlabel('Predicted class', fontsize=11)
    ax.set_ylabel('True class',      fontsize=11)
    ax.set_xticks(range(n_classes))
    ax.set_yticks(range(n_classes))

plt.suptitle('Figure 5 -- Confusion Matrix  |  Pre-trained DNN  |  MNIST test set',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig5_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Most confused pairs
off_diag = [(conf_mat[i, j], i, j)
            for i in range(n_classes) for j in range(n_classes) if i != j]
off_diag.sort(reverse=True)
print('Top-5 confused pairs (true -> predicted, count):')
for cnt, t, p in off_diag[:5]:
    print(f'  {t} -> {p}  :  {cnt} errors')

Top-5 confused pairs (true -> predicted, count):
  9 -> 4  :  29 errors
  7 -> 9  :  23 errors
  5 -> 3  :  23 errors
  4 -> 9  :  22 errors
  5 -> 8  :  19 errors


### 5.8 Train/Test Error vs. Epoch

Beyond cross-entropy loss, tracking the **classification error rate** during training
reveals the generalisation gap and whether the model is overfitting.
We evaluate on both the training subset and the full test set every 20 epochs.

In [27]:
# Architecture and training set fixed for this analysis
dims_err = [784, 200, 200, 10]
EVAL_EVERY = 20
N_EPOCHS   = 200

def train_with_error_tracking(
        dims, X_tr_sub, y_tr_sub, pretrain=True, seed=SEED):
    """
    Train a DNN and record train/test error every EVAL_EVERY epochs.
    Returns (ce_hist, err_train_hist, err_test_hist, epoch_checkpoints).
    """
    dnn = DNN(dims, seed=seed)
    if pretrain:
        print('  Phase 1: DBN pre-training ...')
        dnn.pretrain_DNN(X_tr_sub, **HP_RBM, verbose=False)

    # Backprop epoch by epoch with periodic evaluation
    n          = len(X_tr_sub)
    X_xp       = to_xp(X_tr_sub)
    Y_xp       = to_xp(np.eye(dims[-1], dtype=np.float32)[y_tr_sub])
    eps        = xp.float32(1e-12)
    lr, bs     = HP_BP['lr'], HP_BP['batch_size']
    ce_hist    = []
    err_tr_hist= []
    err_te_hist= []
    checkpoints= []

    print(f'  Phase 2: Backpropagation ({N_EPOCHS} epochs) ...')
    for ep in range(N_EPOCHS):
        perm   = xp.random.permutation(n)
        Xs, Ys = X_xp[perm], Y_xp[perm]
        batch_losses = []
        for i in range(0, n, bs):
            xb = Xs[i:i + bs]; yb = Ys[i:i + bs]; nb = len(xb)
            A, probs = dnn.entree_sortie_reseau(xb)
            loss = -float(xp.mean(
                xp.sum(yb * xp.log(probs + eps), axis=1)))
            batch_losses.append(loss)
            d_out      = probs - yb
            dnn.W_out -= lr * (A[-1].T @ d_out) / nb
            dnn.b_out -= lr * xp.mean(d_out, axis=0, keepdims=True)
            delta = (d_out @ dnn.W_out.T) * dnn._sigmoid_deriv(A[-1])
            for l in range(dnn.L - 1, -1, -1):
                rbm = dnn.dbn.rbms[l]
                rbm.W -= lr * (A[l].T @ delta) / nb
                rbm.b -= lr * xp.mean(delta, axis=0, keepdims=True)
                if l > 0:
                    delta = (delta @ rbm.W.T) * dnn._sigmoid_deriv(A[l])
        ce_hist.append(float(np.mean(batch_losses)))

        if (ep + 1) % EVAL_EVERY == 0 or ep == N_EPOCHS - 1:
            e_tr, _ = dnn.test_DNN(X_tr_sub, y_tr_sub)
            e_te, _ = dnn.test_DNN(Xm_te,    ym_te)
            err_tr_hist.append(e_tr)
            err_te_hist.append(e_te)
            checkpoints.append(ep + 1)
            print(f'    epoch {ep + 1:3d}  CE = {ce_hist[-1]:.5f}  '
                  f'err_train = {e_tr:.4f}  err_test = {e_te:.4f}')

    return (np.array(ce_hist),
            np.array(err_tr_hist),
            np.array(err_te_hist),
            np.array(checkpoints))


# Sub-sample to 10 000 for speed (still representative)
idx_sub  = np.random.RandomState(SEED).permutation(len(Xm_tr))[:10000]
X_sub    = Xm_tr[idx_sub]
y_sub    = ym_tr[idx_sub]

print('Training with error tracking -- pre-trained:')
ce_pre, etr_pre, ete_pre, ck_pre = train_with_error_tracking(
    dims_err, X_sub, y_sub, pretrain=True,  seed=SEED)

print('\nTraining with error tracking -- random init:')
ce_rand, etr_rand, ete_rand, ck_rand = train_with_error_tracking(
    dims_err, X_sub, y_sub, pretrain=False, seed=SEED)

Training with error tracking -- pre-trained:
  Phase 1: DBN pre-training ...
  Phase 2: Backpropagation (200 epochs) ...
    epoch  20  CE = 0.23858  err_train = 0.0629  err_test = 0.0742
    epoch  40  CE = 0.18301  err_train = 0.0473  err_test = 0.0636
    epoch  60  CE = 0.15170  err_train = 0.0373  err_test = 0.0586
    epoch  80  CE = 0.12556  err_train = 0.0304  err_test = 0.0562
    epoch 100  CE = 0.10794  err_train = 0.0257  err_test = 0.0543
    epoch 120  CE = 0.09100  err_train = 0.0207  err_test = 0.0527
    epoch 140  CE = 0.07786  err_train = 0.0165  err_test = 0.0503
    epoch 160  CE = 0.06714  err_train = 0.0131  err_test = 0.0494
    epoch 180  CE = 0.05921  err_train = 0.0101  err_test = 0.0491
    epoch 200  CE = 0.05089  err_train = 0.0082  err_test = 0.0487

Training with error tracking -- random init:
  Phase 2: Backpropagation (200 epochs) ...
    epoch  20  CE = 2.33129  err_train = 0.8990  err_test = 0.8991
    epoch  40  CE = 1.25913  err_train = 0.6651  err

In [28]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: cross-entropy loss
axes[0].plot(range(1, N_EPOCHS + 1), ce_pre,  lw=2,
             color='steelblue', label='Pre-trained (DBN)')
axes[0].plot(range(1, N_EPOCHS + 1), ce_rand, lw=2,
             color='tomato', ls='--', label='Random init')
axes[0].set_title('Cross-Entropy Loss per Epoch', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean CE Loss')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Right: train and test error rates
axes[1].plot(ck_pre,  etr_pre,  '-o', color='steelblue', lw=2, ms=6,
             label='Pre-trained -- train')
axes[1].plot(ck_pre,  ete_pre,  '-o', color='steelblue', lw=2, ms=6,
             ls='--', label='Pre-trained -- test')
axes[1].plot(ck_rand, etr_rand, '-s', color='tomato', lw=2, ms=6,
             label='Random init -- train')
axes[1].plot(ck_rand, ete_rand, '-s', color='tomato', lw=2, ms=6,
             ls='--', label='Random init -- test')
axes[1].set_title('Train / Test Error Rate vs. Epoch', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Classification error rate')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

fig.suptitle(
    'Figure 6 -- Learning curves  |  [784, 200, 200, 10]  |  10 000 training samples\n'
    'Solid = train   Dashed = test',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig6_error_vs_epoch.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pre-trained final:  train={etr_pre[-1]:.4f}  test={ete_pre[-1]:.4f}')
print(f'Random init final:  train={etr_rand[-1]:.4f}  test={ete_rand[-1]:.4f}')

Pre-trained final:  train=0.0082  test=0.0487
Random init final:  train=0.0272  test=0.0777


### 5.9 Analysis of MNIST Results

**Figure 1 (depth):** DBN pre-training is critical for 3 or more hidden layers.
Without it, gradients vanish and error increases monotonically with depth.
The optimal depth for pre-trained networks is 3 layers of 200 neurons each.

**Figure 2 (width):** Both strategies improve up to 400-500 neurons per layer,
then plateau. The pre-trained model outperforms by 0.3-0.6 pp at every width.

**Figure 3 (size):** The gap reaches 5-6 pp at 1 000 samples and shrinks below
0.5 pp at 60 000 samples. Pre-training is most decisive when **labels are scarce**:
the unsupervised DBN phase exploits all pixels without needing any label.

**Figure 4 (misclassifications):** The most common errors involve visually similar digits.
Typical confusion pairs are 4/9, 3/5, and 7/1 -- shapes that are close in pixel space.

**Figure 5 (confusion matrix):** Off-diagonal hot-spots confirm that 4 vs 9
and 3 vs 5 are the hardest pairs. The diagonal is above 0.97 for most classes.

**Figure 6 (learning curves):** The generalisation gap (test - train error)
is small throughout training, indicating that neither model seriously overfits
on the full 60 000 sample training set.

---
## 6. Bonus: Generative Models Comparison <a id='sec6'></a>

Five generative models are trained on MNIST and their sample quality is compared.

| # | Model | Architecture | Training data |
|---|-------|-------------|---------------|
| 6.1 | **RBM** | 784 -> 200 (CD-1) | Binary {0,1} |
| 6.2 | **DBN** | 784 -> 200 -> 100 (greedy) | Binary {0,1} |
| 6.3 | **VAE** | Enc(784->400->20) Dec(10->400->784) | Continuous [0,1] |
| 6.4 | **GAN** | G(100->256->512->784 Tanh) + D | Continuous [-1,1] |
| 6.5 | **DDPM** | U-Net + sinusoidal time embed, T=1000 | Continuous [-1,1] |

In [37]:
# ---------------------------------------------------------------------------
# PyTorch setup and DataLoaders  --  CuPy-safe (zero .numpy() calls)
#
# When CuPy is installed it corrupts NumPy's C-extension so ANY call to
# tensor.numpy() raises "Numpy is not available".
# Fix: stay in pure PyTorch (torch.Tensor) for the entire pipeline.
# ---------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader, TensorDataset

try:
    DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TORCH_OK = True
    print(f'PyTorch {torch.__version__}  |  device = {DEV}')
except Exception as e:
    TORCH_OK = False
    print(f'PyTorch not available: {e}')

if TORCH_OK:
    # Download MNIST (no transform -- we never call .numpy())
    _raw_ds = torchvision.datasets.MNIST(
        '/tmp/mnist_tv', train=True, download=True, transform=None)

    # _raw_ds.data is a uint8 Tensor (60000, 28, 28) -- stay in PyTorch
    _X_raw = _raw_ds.data.float() / 255.0          # float32, range [0, 1]
    _X_raw = _X_raw.unsqueeze(1)                    # (60000, 1, 28, 28)
    _y_raw = _raw_ds.targets                        # long Tensor (60000,)

    # train_loader_vae : [0, 1]  -- BCE loss requires values in (0,1)
    train_loader_vae = DataLoader(
        TensorDataset(_X_raw, _y_raw),
        batch_size=64, shuffle=True,
        num_workers=0, pin_memory=False)
    print(f'train_loader_vae : {len(_X_raw):,} images  range [0, 1]')

    # train_loader : [-1, 1]  -- matches GAN Tanh output and DDPM convention
    _X_11 = _X_raw * 2.0 - 1.0
    train_loader = DataLoader(
        TensorDataset(_X_11, _y_raw),
        batch_size=64, shuffle=True,
        num_workers=0, pin_memory=False)
    print(f'train_loader     : {len(_X_11):,} images  range [-1, 1]')

    print('DataLoaders ready.')

    # ------------------------------------------------------------------
    # t2np : CuPy-safe tensor -> numpy conversion
    # .tolist() converts to plain Python list (zero C-API),
    # then np.array() rebuilds -- does NOT touch the broken C-extension.
    # ------------------------------------------------------------------
    def t2np(tensor):
        """Convert any CPU torch.Tensor to np.ndarray. CuPy-safe."""
        return np.array(tensor.detach().cpu().tolist(), dtype=np.float32)

    print('t2np helper registered (CuPy-safe tensor->numpy).')

PyTorch 1.13.1+cu116  |  device = cuda
train_loader_vae : 60,000 images  range [0, 1]
train_loader     : 60,000 images  range [-1, 1]
DataLoaders ready.
t2np helper registered (CuPy-safe tensor->numpy).


### 6.1 RBM Samples

RBM [784 -> 200] trained with CD-1 on binarised MNIST.
Images are generated by Gibbs sampling (300 steps, 50% burn-in).

In [30]:
# ---------------------------------------------------------------------------
# 6.1  RBM  [784 -> 200]  -- uses existing RBM class (CuPy/NumPy)
# ---------------------------------------------------------------------------
print('Training RBM [784 -> 200] on binary MNIST (50 epochs) ...')
rbm_mnist = RBM(784, 200, seed=SEED)
hist_rbm_mnist = rbm_mnist.train_RBM(
    Xm_tr, n_epochs=50, lr=0.1, batch_size=64, verbose=True)

imgs_rbm = rbm_mnist.generer_image_RBM(n_images=16, n_gibbs=300, seed=10)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for k, ax in enumerate(axes.flatten()):
    ax.imshow(imgs_rbm[k].reshape(28, 28),
              cmap='binary', vmin=0, vmax=1, interpolation='nearest')
    ax.axis('off')
fig.suptitle('RBM [784->200] -- Generated MNIST samples (16 Gibbs chains)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/bonus_rbm_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/bonus_rbm_samples.png')

Training RBM [784 -> 200] on binary MNIST (50 epochs) ...
    [RBM] epoch    1/50  MSE = 0.029360
    [RBM] epoch    6/50  MSE = 0.016876
    [RBM] epoch   11/50  MSE = 0.014536
    [RBM] epoch   16/50  MSE = 0.013471
    [RBM] epoch   21/50  MSE = 0.012896
    [RBM] epoch   26/50  MSE = 0.012421
    [RBM] epoch   31/50  MSE = 0.012137
    [RBM] epoch   36/50  MSE = 0.011952
    [RBM] epoch   41/50  MSE = 0.011728
    [RBM] epoch   46/50  MSE = 0.011589
    [RBM] epoch   50/50  MSE = 0.011521
Saved: figures/bonus_rbm_samples.png


### 6.2 DBN Samples and Reconstruction MSE per Layer

DBN [784 -> 200 -> 100] trained greedily: each RBM receives stochastic
binary activations from the layer below. One MSE curve is recorded per layer.

In [31]:
# ---------------------------------------------------------------------------
# 6.2  DBN  [784 -> 200 -> 100]  -- uses existing DBN class
# ---------------------------------------------------------------------------
print('Training DBN [784->200->100] on binary MNIST (50 epochs/layer) ...')
dbn_mnist = DBN([784, 200, 100], seed=SEED)
dbn_hists = dbn_mnist.train_DBN(
    Xm_tr, n_epochs=50, lr=0.1, batch_size=64, verbose=True)

# MSE curve per layer (one real curve per RBM)
fig, ax = plt.subplots(figsize=(9, 4))
colours = ['steelblue', 'darkorange']
labels  = ['Layer 1  (784 -> 200)', 'Layer 2  (200 -> 100)']
for hist, lbl, col in zip(dbn_hists, labels, colours):
    ax.plot(hist, lw=2, color=col, label=lbl)
ax.set_title(
    'DBN -- Reconstruction MSE per Layer (greedy pre-training)\n'
    'Each curve = one RBM trained on activations from the previous layer.',
    fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Reconstruction MSE')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/bonus_dbn_mse_per_layer.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/bonus_dbn_mse_per_layer.png')

# Generated samples
imgs_dbn = dbn_mnist.generer_image_DBN(n_images=16, n_gibbs=300, seed=11)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for k, ax in enumerate(axes.flatten()):
    ax.imshow(imgs_dbn[k].reshape(28, 28),
              cmap='binary', vmin=0, vmax=1, interpolation='nearest')
    ax.axis('off')
fig.suptitle('DBN [784->200->100] -- Generated MNIST samples (16 chains)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/bonus_dbn_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/bonus_dbn_samples.png')

Training DBN [784->200->100] on binary MNIST (50 epochs/layer) ...

  [DBN] Training layer 1/2
        Input dim: 784   Hidden dim: 200
    [RBM] epoch    1/50  MSE = 0.029396
    [RBM] epoch    6/50  MSE = 0.016874
    [RBM] epoch   11/50  MSE = 0.014554
    [RBM] epoch   16/50  MSE = 0.013545
    [RBM] epoch   21/50  MSE = 0.012984
    [RBM] epoch   26/50  MSE = 0.012598
    [RBM] epoch   31/50  MSE = 0.012324
    [RBM] epoch   36/50  MSE = 0.012065
    [RBM] epoch   41/50  MSE = 0.011944
    [RBM] epoch   46/50  MSE = 0.011765
    [RBM] epoch   50/50  MSE = 0.011708

  [DBN] Training layer 2/2
        Input dim: 200   Hidden dim: 100
    [RBM] epoch    1/50  MSE = 0.098970
    [RBM] epoch    6/50  MSE = 0.066061
    [RBM] epoch   11/50  MSE = 0.061066
    [RBM] epoch   16/50  MSE = 0.058738
    [RBM] epoch   21/50  MSE = 0.057270
    [RBM] epoch   26/50  MSE = 0.056274
    [RBM] epoch   31/50  MSE = 0.055549
    [RBM] epoch   36/50  MSE = 0.055002
    [RBM] epoch   41/50  MSE = 0.05

### 6.3 Variational Autoencoder (VAE)

Architecture:

- **Encoder:** Linear(784, 400) -> ReLU -> Linear(400, 20)  
  The 20-dim output is split into `mu` (10) and `log_var` (10) via `.chunk(2, dim=1)`.
- **Decoder:** Linear(10, 400) -> ReLU -> Linear(400, 784) -> Sigmoid
- **Loss:** BCE (reconstruction) + KL divergence
- **Training:** Each batch is normalised to [0, 1] inside the loop
  (images from `train_loader` are in [-1, 1], shifted to [0, 1] per batch).

In [38]:
if TORCH_OK:
    class VAE(nn.Module):
        def __init__(self):
            super(VAE, self).__init__()
            self.encoder = nn.Sequential(
                nn.Linear(784, 400),
                nn.ReLU(),
                nn.Linear(400, 20)
            )
            self.decoder = nn.Sequential(
                nn.Linear(10, 400),
                nn.ReLU(),
                nn.Linear(400, 784),
                nn.Sigmoid()
            )

        def forward(self, x):
            x = x.view(-1, 784)
            z_mean, z_log_var = self.encoder(x).chunk(2, dim=1)
            z = z_mean + torch.exp(0.5 * z_log_var) * torch.randn_like(z_mean)
            x_reconstructed = self.decoder(z)
            return x_reconstructed, z_mean, z_log_var


    def vae_loss(x_reconstructed, x, z_mean, z_log_var):
        N = x.size(0)
        bce = nn.functional.binary_cross_entropy(
            x_reconstructed, x.view(-1, 784), reduction='sum') / N
        kld = -0.5 * torch.sum(
            1 + z_log_var - z_mean.pow(2) - z_log_var.exp()) / N
        return bce + kld


    def train_vae(vae, epochs=200):
        optimizer = optim.Adam(vae.parameters(), lr=1e-3)
        loss_hist = []
        for epoch in range(epochs):
            total_loss = 0.0
            for images, _ in train_loader_vae:
                images = images.view(-1, 784).to(DEV)
                x_reconstructed, z_mean, z_log_var = vae(images)
                loss = vae_loss(x_reconstructed, images, z_mean, z_log_var)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            avg = total_loss / len(train_loader_vae)
            loss_hist.append(avg)
            if (epoch + 1) % 20 == 0:
                print(f'  Epoch [{epoch + 1}/{epochs}]  ELBO/sample: {avg:.2f}')
        return loss_hist


    def generate_vae_images(vae, num_samples=16):
        """Sample z ~ N(0,I) and decode to images in [0,1]. CuPy-safe."""
        vae.eval()
        z = torch.randn(num_samples, 10).to(DEV)
        with torch.no_grad():
            generated = vae.decoder(z).cpu().view(-1, 28, 28)
        return t2np(generated)  # never calls .numpy() -- CuPy-safe


    vae = VAE().to(DEV)
    print(f'VAE parameters: {sum(p.numel() for p in vae.parameters()):,}')
    print('\nTraining VAE (200 epochs) ...')
    vae_loss_hist = train_vae(vae, epochs=200)

    imgs_vae = generate_vae_images(vae, num_samples=16)

    fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
    for k, ax in enumerate(axes.flatten()):
        ax.imshow(imgs_vae[k], cmap='gray', vmin=0, vmax=1,
                  interpolation='nearest')
        ax.axis('off')
    fig.suptitle('VAE [z=10] -- Generated MNIST samples',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/bonus_vae_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/bonus_vae_samples.png')

VAE parameters: 640,804

Training VAE (200 epochs) ...
  Epoch [20/200]  ELBO/sample: 107.10
  Epoch [40/200]  ELBO/sample: 105.05
  Epoch [60/200]  ELBO/sample: 104.13
  Epoch [80/200]  ELBO/sample: 103.63
  Epoch [100/200]  ELBO/sample: 103.26
  Epoch [120/200]  ELBO/sample: 102.95
  Epoch [140/200]  ELBO/sample: 102.73
  Epoch [160/200]  ELBO/sample: 102.51
  Epoch [180/200]  ELBO/sample: 102.38
  Epoch [200/200]  ELBO/sample: 102.28
Saved: figures/bonus_vae_samples.png


### 6.4 Generative Adversarial Network (GAN)

Architecture:

- **Generator:** Linear(100,256) -> ReLU -> Linear(256,512) -> ReLU -> Linear(512,784) -> **Tanh** -> view(-1,1,28,28)  
  Output in [-1, 1]. Rescaled to [0, 1] for display using `(x + 1) / 2`.
- **Discriminator:** x.view(-1,784) -> Linear(784,512) -> LeakyReLU(0.2) -> Linear(512,256) -> LeakyReLU(0.2) -> Linear(256,1) -> Sigmoid
- **Loss:** Binary cross-entropy (minimax objective)
- **Training data:** `train_loader` with images normalised to [-1, 1]

In [39]:
if TORCH_OK:
    class Generator(nn.Module):
        def __init__(self):
            super(Generator, self).__init__()
            self.model = nn.Sequential(
                nn.Linear(100, 256), nn.ReLU(),
                nn.Linear(256, 512), nn.ReLU(),
                nn.Linear(512, 784), nn.Tanh()
            )
        def forward(self, z):
            return self.model(z).view(-1, 1, 28, 28)

    class Discriminator(nn.Module):
        def __init__(self):
            super(Discriminator, self).__init__()
            self.model = nn.Sequential(
                nn.Linear(784, 512), nn.LeakyReLU(0.2),
                nn.Linear(512, 256), nn.LeakyReLU(0.2),
                nn.Linear(256, 1),   nn.Sigmoid()
            )
        def forward(self, x):
            return self.model(x.view(-1, 784))

    def train_gan(generator, discriminator, epochs=200):
        loss_fn     = nn.BCELoss()
        optimizer_g = optim.Adam(generator.parameters(),     lr=0.0002)
        optimizer_d = optim.Adam(discriminator.parameters(), lr=0.0002)
        for epoch in range(epochs):
            for real_images, _ in train_loader:
                bs = real_images.shape[0]
                real_images = real_images.to(DEV)
                z = torch.randn(bs, 100).to(DEV)
                fake_images = generator(z)
                real_labels = torch.ones(bs,  1).to(DEV)
                fake_labels = torch.zeros(bs, 1).to(DEV)
                loss_d = (loss_fn(discriminator(real_images), real_labels) +
                          loss_fn(discriminator(fake_images.detach()), fake_labels))
                optimizer_d.zero_grad(); loss_d.backward(); optimizer_d.step()
                loss_g = loss_fn(discriminator(generator(torch.randn(bs,100).to(DEV))),
                                 real_labels)
                optimizer_g.zero_grad(); loss_g.backward(); optimizer_g.step()
            if (epoch + 1) % 20 == 0:
                print(f'  Epoch [{epoch+1}/200]  loss_D={loss_d.item():.4f}  loss_G={loss_g.item():.4f}')

    def generate_gan_images(generator, num_samples=16):
        """Generate and rescale from [-1,1] to [0,1]. CuPy-safe."""
        generator.eval()
        z = torch.randn(num_samples, 100).to(DEV)
        with torch.no_grad():
            fake = generator(z).cpu()
        fake = (fake + 1) / 2                   # [-1,1] -> [0,1]
        return t2np(fake.squeeze())             # CuPy-safe, no .numpy()

    generator     = Generator().to(DEV)
    discriminator = Discriminator().to(DEV)
    print(f'Generator  {sum(p.numel() for p in generator.parameters()):,} params')
    print(f'Discriminator  {sum(p.numel() for p in discriminator.parameters()):,} params')
    print('\nTraining GAN (200 epochs) ...')
    train_gan(generator, discriminator, epochs=200)

    imgs_gan = generate_gan_images(generator, num_samples=16)

    fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
    for k, ax in enumerate(axes.flatten()):
        ax.imshow(imgs_gan[k], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        ax.axis('off')
    fig.suptitle('GAN [z=100, Tanh] -- Generated MNIST samples',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/bonus_gan_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/bonus_gan_samples.png')

Generator  559,632 params
Discriminator  533,505 params

Training GAN (200 epochs) ...
  Epoch [20/200]  loss_D=0.4148  loss_G=2.8482
  Epoch [40/200]  loss_D=0.6482  loss_G=1.6947
  Epoch [60/200]  loss_D=0.8717  loss_G=2.0092
  Epoch [80/200]  loss_D=0.9160  loss_G=1.9270
  Epoch [100/200]  loss_D=1.1909  loss_G=1.2944
  Epoch [120/200]  loss_D=0.8157  loss_G=1.2978
  Epoch [140/200]  loss_D=1.1356  loss_G=1.7363
  Epoch [160/200]  loss_D=1.0549  loss_G=1.4492
  Epoch [180/200]  loss_D=1.2464  loss_G=1.0402
  Epoch [200/200]  loss_D=0.8991  loss_G=1.4204
Saved: figures/bonus_gan_samples.png


### 6.5 Denoising Diffusion Probabilistic Model (DDPM)

Full from-scratch implementation following [Ho et al., NeurIPS 2020](https://arxiv.org/abs/2006.11239).

**Forward process** -- gradually corrupts $x_0$ over $T=1000$ steps:
$$q(x_t|x_0) = \mathcal{N}(x_t;\,\sqrt{\bar\alpha_t}\,x_0,\,(1-\bar\alpha_t)I)$$
where $\alpha_t = 1-\beta_t$ and $\bar\alpha_t = \prod_{s=1}^t \alpha_s$,
with a linear schedule $\beta_t \in [10^{-4},\,0.02]$.

**Reverse process** -- recovers $x_{t-1}$ from $x_t$ using the U-Net noise prediction $\epsilon_\theta$:
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t,t)\right) + \sigma_t z, \quad z\sim\mathcal{N}(0,I)$$

**Architecture** -- U-Net with:
- Sinusoidal time embedding projected through 2-layer MLP
- 3 DownC encoder blocks with ResNet + GroupNorm + SiLU + Self-Attention + Downsample
- 2 MidC bottleneck blocks
- 3 UpC decoder blocks with skip connections from encoder
- Training objective: $\mathcal{L} = \mathbb{E}_{t,x_0,\epsilon}\|\epsilon - \epsilon_\theta(x_t,t)\|^2$

In [40]:
if TORCH_OK:
    # ===========================================================================
    # DDPM -- Forward Process (no trainable parameters)
    # Precomputes beta, alpha, alpha_bar schedules at construction time.
    # ===========================================================================
    class DiffusionForwardProcess:
        """
        Forward diffusion process from Ho et al. (2020).
        Closed-form sampling: x_t = sqrt(abar_t)*x0 + sqrt(1-abar_t)*eps.
        """
        def __init__(self, num_time_steps=1000,
                     beta_start=1e-4, beta_end=0.02):
            self.betas                     = torch.linspace(
                beta_start, beta_end, num_time_steps)
            self.alphas                    = 1 - self.betas
            self.alpha_bars                = torch.cumprod(self.alphas, dim=0)
            self.sqrt_alpha_bars           = torch.sqrt(self.alpha_bars)
            self.sqrt_one_minus_alpha_bars = torch.sqrt(1 - self.alpha_bars)

        def add_noise(self, original, noise, t):
            """
            Add noise to a batch at timestep t.
            :param original : Tensor (B,C,H,W) clean images in [-1,1]
            :param noise    : Tensor (B,C,H,W) ~ N(0,I)
            :param t        : Tensor (B,)      integer timesteps
            """
            s_ab   = self.sqrt_alpha_bars.to(original.device)[t][:, None, None, None]
            s_1mab = self.sqrt_one_minus_alpha_bars.to(original.device)[t][:, None, None, None]
            return s_ab * original + s_1mab * noise


    # ===========================================================================
    # DDPM -- Reverse Process
    # ===========================================================================
    class DiffusionReverseProcess:
        """
        Reverse diffusion: recover x_{t-1} from x_t and predicted noise.
        At t=0 returns the mean without variance noise.
        """
        def __init__(self, num_time_steps=1000,
                     beta_start=1e-4, beta_end=0.02):
            self.b     = torch.linspace(beta_start, beta_end, num_time_steps)
            self.a     = 1 - self.b
            self.a_bar = torch.cumprod(self.a, dim=0)

        def sample_prev_timestep(self, xt, noise_pred, t):
            """
            :param xt         : Tensor (B,C,H,W) noisy image at step t
            :param noise_pred : Tensor (B,C,H,W) predicted noise from U-Net
            :param t          : int scalar
            :returns (x_prev, x0_pred)
            """
            dev   = xt.device
            a_bar = self.a_bar.to(dev)
            a     = self.a.to(dev)
            b     = self.b.to(dev)

            # Predicted clean image
            x0 = (xt - torch.sqrt(1 - a_bar[t]) * noise_pred)
            x0 = x0 / torch.sqrt(a_bar[t])
            x0 = torch.clamp(x0, -1., 1.)

            # Posterior mean
            mean = (xt - (1 - a[t]) * noise_pred
                    / torch.sqrt(1 - a_bar[t])) / torch.sqrt(a[t])

            if t == 0:
                return mean, x0

            variance = (1 - a_bar[t - 1]) / (1 - a_bar[t]) * b[t]
            sigma    = variance ** 0.5
            z        = torch.randn(xt.shape).to(dev)
            return mean + sigma * z, x0


    print('DiffusionForwardProcess and DiffusionReverseProcess defined.')

DiffusionForwardProcess and DiffusionReverseProcess defined.


In [41]:
if TORCH_OK:
    # ===========================================================================
    # Time Embedding (sinusoidal, from ddpm-from-scratch reference)
    # ===========================================================================
    def get_time_embedding(time_steps: torch.Tensor, t_emb_dim: int) -> torch.Tensor:
        """
        Sinusoidal time embedding identical to Transformer positional encoding.
        :param time_steps : (B,) integer timesteps
        :param t_emb_dim  : embedding dimension (must be even)
        :return           : (B, t_emb_dim)
        """
        assert t_emb_dim % 2 == 0, 'Time embedding dim must be even.'
        factor  = 2 * torch.arange(
            0, t_emb_dim // 2, dtype=torch.float32,
            device=time_steps.device) / t_emb_dim
        factor  = 10000 ** factor
        t_emb   = time_steps[:, None].float() / factor
        return torch.cat([torch.sin(t_emb), torch.cos(t_emb)], dim=1)


    # ===========================================================================
    # U-Net utility modules (exact reference implementation)
    # ===========================================================================
    class NormActConv(nn.Module):
        """GroupNorm -> SiLU -> Conv2d."""
        def __init__(self, in_channels, out_channels,
                     num_groups=8, kernel_size=3, norm=True, act=True):
            super(NormActConv, self).__init__()
            self.g_norm = (nn.GroupNorm(num_groups, in_channels)
                           if norm else nn.Identity())
            self.act    = nn.SiLU() if act else nn.Identity()
            self.conv   = nn.Conv2d(
                in_channels, out_channels, kernel_size,
                padding=(kernel_size - 1) // 2)

        def forward(self, x):
            return self.conv(self.act(self.g_norm(x)))


    class TimeEmbedding(nn.Module):
        """Project time embedding to required channel dimension."""
        def __init__(self, n_out, t_emb_dim=128):
            super(TimeEmbedding, self).__init__()
            self.te_block = nn.Sequential(
                nn.SiLU(), nn.Linear(t_emb_dim, n_out))

        def forward(self, x):
            return self.te_block(x)


    class SelfAttentionBlock(nn.Module):
        """GroupNorm + multi-head self-attention."""
        def __init__(self, num_channels, num_groups=8, num_heads=4, norm=True):
            super(SelfAttentionBlock, self).__init__()
            self.g_norm = (nn.GroupNorm(num_groups, num_channels)
                           if norm else nn.Identity())
            self.attn   = nn.MultiheadAttention(
                num_channels, num_heads, batch_first=True)

        def forward(self, x):
            B, C, H, W = x.shape
            x = x.reshape(B, C, H * W)
            x = self.g_norm(x)
            x = x.transpose(1, 2)
            x, _ = self.attn(x, x, x)
            return x.transpose(1, 2).reshape(B, C, H, W)


    class Downsample(nn.Module):
        """
        Spatial downsampling by factor k.
        Combines strided conv (out//2 ch) and MaxPool (out//2 ch),
        concatenated to produce out_channels.
        """
        def __init__(self, in_channels, out_channels,
                     k=2, use_conv=True, use_mpool=True):
            super(Downsample, self).__init__()
            self.use_conv  = use_conv
            self.use_mpool = use_mpool
            self.cv = nn.Sequential(
                nn.Conv2d(in_channels, in_channels, kernel_size=1),
                nn.Conv2d(in_channels,
                          out_channels // 2 if use_mpool else out_channels,
                          kernel_size=4, stride=k, padding=1)
            ) if use_conv else nn.Identity()
            self.mpool = nn.Sequential(
                nn.MaxPool2d(k, k),
                nn.Conv2d(in_channels,
                          out_channels // 2 if use_conv else out_channels,
                          kernel_size=1)
            ) if use_mpool else nn.Identity()

        def forward(self, x):
            if not self.use_conv:  return self.mpool(x)
            if not self.use_mpool: return self.cv(x)
            return torch.cat([self.cv(x), self.mpool(x)], dim=1)


    class Upsample(nn.Module):
        """
        Spatial upsampling by factor k.
        Combines ConvTranspose (out//2 ch) and bilinear Upsample (out//2 ch),
        concatenated to produce out_channels.
        """
        def __init__(self, in_channels, out_channels,
                     k=2, use_conv=True, use_upsample=True):
            super(Upsample, self).__init__()
            self.use_conv     = use_conv
            self.use_upsample = use_upsample
            self.cv = nn.Sequential(
                nn.ConvTranspose2d(
                    in_channels,
                    out_channels // 2 if use_upsample else out_channels,
                    kernel_size=4, stride=k, padding=1),
                nn.Conv2d(
                    out_channels // 2 if use_upsample else out_channels,
                    out_channels // 2 if use_upsample else out_channels,
                    kernel_size=1)
            ) if use_conv else nn.Identity()
            self.up = nn.Sequential(
                nn.Upsample(scale_factor=k, mode='bilinear', align_corners=False),
                nn.Conv2d(in_channels,
                          out_channels // 2 if use_conv else out_channels,
                          kernel_size=1)
            ) if use_upsample else nn.Identity()

        def forward(self, x):
            if not self.use_conv:     return self.up(x)
            if not self.use_upsample: return self.cv(x)
            return torch.cat([self.cv(x), self.up(x)], dim=1)


    print('Time embedding and U-Net utility modules defined.')

Time embedding and U-Net utility modules defined.


In [42]:
if TORCH_OK:
    # ===========================================================================
    # DownC -- encoder block (from ddpm-from-scratch reference, cell 20)
    # ===========================================================================
    class DownC(nn.Module):
        """
        Down-convolution block:
        1. Conv1 + TimeEmbedding  (ResNet input)
        2. Conv2
        3. Skip connection from step 1 input
        4. Self-Attention (residual)
        5. Optional Downsample
        """
        def __init__(self, in_channels, out_channels,
                     t_emb_dim=128, num_layers=2, down_sample=True):
            super(DownC, self).__init__()
            self.num_layers = num_layers
            self.conv1 = nn.ModuleList([
                NormActConv(in_channels if i == 0 else out_channels, out_channels)
                for i in range(num_layers)])
            self.conv2      = nn.ModuleList([
                NormActConv(out_channels, out_channels)
                for _ in range(num_layers)])
            self.te_block   = nn.ModuleList([
                TimeEmbedding(out_channels, t_emb_dim)
                for _ in range(num_layers)])
            self.attn_block = nn.ModuleList([
                SelfAttentionBlock(out_channels)
                for _ in range(num_layers)])
            self.down_block = (Downsample(out_channels, out_channels)
                               if down_sample else nn.Identity())
            self.res_block  = nn.ModuleList([
                nn.Conv2d(in_channels if i == 0 else out_channels,
                          out_channels, kernel_size=1)
                for i in range(num_layers)])

        def forward(self, x, t_emb):
            out = x
            for i in range(self.num_layers):
                resnet_input = out
                out = self.conv1[i](out)
                out = out + self.te_block[i](t_emb)[:, :, None, None]
                out = self.conv2[i](out)
                out = out + self.res_block[i](resnet_input)  # skip
                out = out + self.attn_block[i](out)           # attention
            return self.down_block(out)


    # ===========================================================================
    # MidC -- bottleneck block (from reference, cell 22)
    # ===========================================================================
    class MidC(nn.Module):
        """
        Bottleneck block: ResNet -> (Attention + ResNet) x num_layers.
        """
        def __init__(self, in_channels, out_channels,
                     t_emb_dim=128, num_layers=2):
            super(MidC, self).__init__()
            self.num_layers  = num_layers
            self.conv1       = nn.ModuleList([
                NormActConv(in_channels if i == 0 else out_channels, out_channels)
                for i in range(num_layers + 1)])
            self.conv2       = nn.ModuleList([
                NormActConv(out_channels, out_channels)
                for _ in range(num_layers + 1)])
            self.te_block    = nn.ModuleList([
                TimeEmbedding(out_channels, t_emb_dim)
                for _ in range(num_layers + 1)])
            self.attn_block  = nn.ModuleList([
                SelfAttentionBlock(out_channels)
                for _ in range(num_layers)])
            self.res_block   = nn.ModuleList([
                nn.Conv2d(in_channels if i == 0 else out_channels,
                          out_channels, kernel_size=1)
                for i in range(num_layers + 1)])

        def forward(self, x, t_emb):
            out = x
            # First ResNet block
            resnet_input = out
            out = self.conv1[0](out)
            out = out + self.te_block[0](t_emb)[:, :, None, None]
            out = self.conv2[0](out)
            out = out + self.res_block[0](resnet_input)
            # Alternating attention + ResNet
            for i in range(self.num_layers):
                out          = out + self.attn_block[i](out)
                resnet_input = out
                out = self.conv1[i + 1](out)
                out = out + self.te_block[i + 1](t_emb)[:, :, None, None]
                out = self.conv2[i + 1](out)
                out = out + self.res_block[i + 1](resnet_input)
            return out


    # ===========================================================================
    # UpC -- decoder block (from reference, cell 24)
    # ===========================================================================
    class UpC(nn.Module):
        """
        Up-convolution block:
        1. Optional Upsample (in_ch -> in_ch//2)
        2. Cat with skip connection from encoder
        3. ResNet conv + TimeEmbedding + Self-Attention
        """
        def __init__(self, in_channels, out_channels,
                     t_emb_dim=128, num_layers=2, up_sample=True):
            super(UpC, self).__init__()
            self.num_layers  = num_layers
            self.up_block    = (Upsample(in_channels, in_channels // 2)
                                if up_sample else nn.Identity())
            self.conv1       = nn.ModuleList([
                NormActConv(in_channels if i == 0 else out_channels, out_channels)
                for i in range(num_layers)])
            self.conv2       = nn.ModuleList([
                NormActConv(out_channels, out_channels)
                for _ in range(num_layers)])
            self.te_block    = nn.ModuleList([
                TimeEmbedding(out_channels, t_emb_dim)
                for _ in range(num_layers)])
            self.attn_block  = nn.ModuleList([
                SelfAttentionBlock(out_channels)
                for _ in range(num_layers)])
            self.res_block   = nn.ModuleList([
                nn.Conv2d(in_channels if i == 0 else out_channels,
                          out_channels, kernel_size=1)
                for i in range(num_layers)])

        def forward(self, x, down_out, t_emb):
            x   = self.up_block(x)                    # upsample
            x   = torch.cat([x, down_out], dim=1)     # concat skip
            out = x
            for i in range(self.num_layers):
                resnet_input = out
                out = self.conv1[i](out)
                out = out + self.te_block[i](t_emb)[:, :, None, None]
                out = self.conv2[i](out)
                out = out + self.res_block[i](resnet_input)
                out = out + self.attn_block[i](out)
            return out


    print('DownC, MidC, UpC defined.')

DownC, MidC, UpC defined.


In [43]:
if TORCH_OK:
    # ===========================================================================
    # U-Net (exact reference implementation, ddpm-from-scratch cell 26)
    #
    # Channel trace (28x28 input):
    #   cv1 :  1ch  ->  32ch  @28x28
    #   down[0]: 32  ->  64ch  @14x14  (downsample=True)
    #   down[1]: 64  -> 128ch  @ 7x7   (downsample=True)
    #   down[2]: 128 -> 256ch  @ 7x7   (downsample=False)
    #   mid[0] : 256 -> 256ch  @ 7x7
    #   mid[1] : 256 -> 128ch  @ 7x7
    #   up[0]  : 256 -> 128ch  @ 7x7   (up_sample=False, cat skip=128@7x7)
    #   up[1]  : 128 ->  64ch  @14x14  (up_sample=True,  cat skip=64@14x14)
    #   up[2]  :  64 ->  16ch  @28x28  (up_sample=True,  cat skip=32@28x28)
    #   cv2 :  16ch ->   1ch  @28x28
    #
    # Skip connections: down_outs appended BEFORE each down block.
    # ===========================================================================
    class Unet(nn.Module):
        def __init__(self,
                     im_channels:  int  = 1,
                     down_ch:      list = [32, 64, 128, 256],
                     mid_ch:       list = [256, 256, 128],
                     up_ch:        list = [256, 128, 64, 16],
                     down_sample:  list = [True, True, False],
                     t_emb_dim:    int  = 128,
                     num_downc_layers: int = 2,
                     num_midc_layers:  int = 2,
                     num_upc_layers:   int = 2):
            super(Unet, self).__init__()
            self.t_emb_dim  = t_emb_dim
            self.up_sample  = list(reversed(down_sample))  # [False, True, True]

            # Initial convolution
            self.cv1 = nn.Conv2d(im_channels, down_ch[0], kernel_size=3, padding=1)

            # Time embedding projection MLP
            self.t_proj = nn.Sequential(
                nn.Linear(t_emb_dim, t_emb_dim),
                nn.SiLU(),
                nn.Linear(t_emb_dim, t_emb_dim))

            # Encoder
            self.downs = nn.ModuleList([
                DownC(down_ch[i], down_ch[i + 1],
                      t_emb_dim, num_downc_layers, down_sample[i])
                for i in range(len(down_ch) - 1)])

            # Bottleneck
            self.mids = nn.ModuleList([
                MidC(mid_ch[i], mid_ch[i + 1],
                     t_emb_dim, num_midc_layers)
                for i in range(len(mid_ch) - 1)])

            # Decoder
            self.ups = nn.ModuleList([
                UpC(up_ch[i], up_ch[i + 1],
                    t_emb_dim, num_upc_layers, self.up_sample[i])
                for i in range(len(up_ch) - 1)])

            # Final convolution
            self.cv2 = nn.Sequential(
                nn.GroupNorm(8, up_ch[-1]),
                nn.Conv2d(up_ch[-1], im_channels, kernel_size=3, padding=1))

        def forward(self, x, t):
            out   = self.cv1(x)
            t_emb = get_time_embedding(t, self.t_emb_dim)
            t_emb = self.t_proj(t_emb)

            # Encoder: store skip connections BEFORE each down block
            down_outs = []
            for down in self.downs:
                down_outs.append(out)       # save current feature map
                out = down(out, t_emb)      # apply down block

            # Bottleneck
            for mid in self.mids:
                out = mid(out, t_emb)

            # Decoder: consume skips in LIFO order (pop = last-in-first-out)
            for up in self.ups:
                down_out = down_outs.pop()
                out      = up(out, down_out, t_emb)

            return self.cv2(out)   # (B, 1, 28, 28)


    # Sanity check
    _x = torch.randn(2, 1, 28, 28).to(DEV)
    _t = torch.randint(0, 1000, (2,)).to(DEV)
    _u = Unet().to(DEV)
    _o = _u(_x, _t)
    assert _o.shape == (2, 1, 28, 28), f'Unexpected output shape: {_o.shape}'
    print(f'U-Net output shape: {_o.shape}  [OK]')
    print(f'U-Net parameters  : {sum(p.numel() for p in _u.parameters()):,}')
    del _x, _t, _u, _o

U-Net output shape: torch.Size([2, 1, 28, 28])  [OK]
U-Net parameters  : 10,846,433


In [44]:
if TORCH_OK:
    # ===========================================================================
    # DDPM Training loop (from reference, cell 33 adapted for torchvision MNIST)
    # Objective: MSE between true noise eps and predicted noise eps_theta(x_t, t)
    # ===========================================================================
    NUM_TIMESTEPS = 1000
    DDPM_EPOCHS   = 30        # sufficient on H100 for recognisable digits
    DDPM_LR       = 1e-4
    MODEL_PATH    = 'ddpm_unet.pth'

    model_ddpm = Unet().to(DEV)
    dfp        = DiffusionForwardProcess(num_time_steps=NUM_TIMESTEPS)
    optimizer  = torch.optim.Adam(model_ddpm.parameters(), lr=DDPM_LR)
    criterion  = nn.MSELoss()
    print(f'DDPM U-Net parameters: {sum(p.numel() for p in model_ddpm.parameters()):,}')

    best_loss      = float('inf')
    ddpm_loss_hist = []

    print(f'\nTraining DDPM ({DDPM_EPOCHS} epochs, '
          f'train_loader images in [-1,1]) ...')
    for epoch in range(DDPM_EPOCHS):
        losses = []
        model_ddpm.train()

        for imgs, _ in train_loader:
            imgs  = imgs.to(DEV)                         # (B,1,28,28) in [-1,1]
            noise = torch.randn_like(imgs)
            t     = torch.randint(0, NUM_TIMESTEPS,
                                  (imgs.shape[0],), device=DEV)

            noisy_imgs = dfp.add_noise(imgs, noise, t)   # forward process

            optimizer.zero_grad()
            noise_pred = model_ddpm(noisy_imgs, t)        # U-Net noise prediction
            loss       = criterion(noise_pred, noise)     # MSE objective
            loss.backward()
            # Gradient clipping: prevents exploding gradients (NaN divergence)
            torch.nn.utils.clip_grad_norm_(
                model_ddpm.parameters(), max_norm=1.0)
            optimizer.step()
            val = loss.item()
            if not (val != val):   # skip NaN batches
                losses.append(val)

        mean_loss = float(np.mean(losses))
        ddpm_loss_hist.append(mean_loss)
        print(f'  Epoch {epoch + 1:3d}/{DDPM_EPOCHS}  Loss: {mean_loss:.4f}')

        # Save best model
        if mean_loss < best_loss:
            best_loss = mean_loss
            torch.save(model_ddpm, MODEL_PATH)

    print(f'\nDone. Best loss: {best_loss:.4f}  Saved: {MODEL_PATH}')

    # Training loss curve
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(ddpm_loss_hist, lw=2, color='steelblue')
    ax.set_title('DDPM U-Net -- Training Loss (noise prediction MSE)', fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('figures/bonus_ddpm_loss.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/bonus_ddpm_loss.png')

DDPM U-Net parameters: 10,846,433

Training DDPM (30 epochs, train_loader images in [-1,1]) ...
  Epoch   1/30  Loss: 0.0786
  Epoch   2/30  Loss: 0.0344
  Epoch   3/30  Loss: 0.0300
  Epoch   4/30  Loss: 0.0278
  Epoch   5/30  Loss: 0.0266
  Epoch   6/30  Loss: 0.0258
  Epoch   7/30  Loss: 0.0250
  Epoch   8/30  Loss: 0.0248
  Epoch   9/30  Loss: 0.0243
  Epoch  10/30  Loss: 0.0241
  Epoch  11/30  Loss: 0.0239
  Epoch  12/30  Loss: 0.0234
  Epoch  13/30  Loss: 0.0234
  Epoch  14/30  Loss: 0.0233
  Epoch  15/30  Loss: 0.0231
  Epoch  16/30  Loss: 0.0230
  Epoch  17/30  Loss: 0.0230
  Epoch  18/30  Loss: 0.0228
  Epoch  19/30  Loss: 0.0228
  Epoch  20/30  Loss: 0.0227
  Epoch  21/30  Loss: 0.0224
  Epoch  22/30  Loss: 0.0223
  Epoch  23/30  Loss: 0.0222
  Epoch  24/30  Loss: 0.0222
  Epoch  25/30  Loss: 0.0222
  Epoch  26/30  Loss: 0.0221
  Epoch  27/30  Loss: 0.0221
  Epoch  28/30  Loss: 0.0223
  Epoch  29/30  Loss: 0.0220
  Epoch  30/30  Loss: 0.0221

Done. Best loss: 0.0220  Saved: d

In [46]:
if TORCH_OK:
    def generate_ddpm(model_path, num_samples=16,
                      num_timesteps=1000, img_size=28, in_channels=1):
        """
        Generate images by running the full reverse diffusion process.
        Returns np.ndarray (num_samples, 28, 28) in [0, 1]. CuPy-safe.
        """
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        drp    = DiffusionReverseProcess(num_time_steps=num_timesteps)
        model  = torch.load(model_path, map_location=device)
        model.eval()

        generated = []
        for _ in range(num_samples):
            xt = torch.randn(1, in_channels, img_size, img_size).to(device)
            with torch.no_grad():
                for t in reversed(range(num_timesteps)):
                    noise_pred = model(xt, torch.as_tensor(t).unsqueeze(0).to(device))
                    xt, _ = drp.sample_prev_timestep(
                        xt, noise_pred, torch.as_tensor(t).to(device))
            xt = torch.clamp(xt, -1., 1.).detach().cpu()
            xt = (xt + 1) / 2
            generated.append(t2np(xt[0][0]))  # CuPy-safe, no .numpy()

        return np.stack(generated)  # (num_samples, 28, 28)


    print(f'Generating DDPM samples ({NUM_TIMESTEPS} steps per image) ...')
    imgs_ddpm = generate_ddpm(
        MODEL_PATH, num_samples=16,
        num_timesteps=NUM_TIMESTEPS, img_size=28, in_channels=1)

    fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
    for k, ax in enumerate(axes.flatten()):
        ax.imshow(imgs_ddpm[k], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        ax.axis('off')
    fig.suptitle(f'DDPM [U-Net, T={NUM_TIMESTEPS}] -- Generated MNIST samples\n'
                 'Starting from Gaussian noise, full reverse diffusion.',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/bonus_ddpm_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/bonus_ddpm_samples.png')

Generating DDPM samples (1000 steps per image) ...
Saved: figures/bonus_ddpm_samples.png


### 6.6 Side-by-Side Comparison

All five models displayed on the same figure for direct visual comparison.
16 samples per model, 2 rows of 8 columns each.

In [47]:
if TORCH_OK:
    model_data = [
        ('RBM   [784->200]\n(binary, Gibbs sampling)',
         imgs_rbm.reshape(16, 28, 28),    'binary'),
        ('DBN   [784->200->100]\n(binary, greedy + Gibbs)',
         imgs_dbn.reshape(16, 28, 28),    'binary'),
        ('VAE   [z=10]\n(continuous [0,1], ELBO)',
         imgs_vae,                         'gray'),
        ('GAN   [z=100, Tanh]\n(continuous [-1,1], adversarial)',
         imgs_gan,                         'gray'),
        ('DDPM  [U-Net, T=1000]\n(reverse diffusion)',
         imgs_ddpm,                        'gray'),
    ]
    N = 16

    fig, axes = plt.subplots(5, N, figsize=(N * 1.4, 9.5))
    for row, (name, imgs, cmap) in enumerate(model_data):
        axes[row, 0].set_ylabel(
            name, fontsize=9, fontweight='bold',
            rotation=0, labelpad=140, va='center')
        for col in range(N):
            axes[row, col].imshow(
                imgs[col], cmap=cmap, vmin=0, vmax=1,
                interpolation='nearest')
            axes[row, col].axis('off')

    fig.suptitle(
        'Generative Models Comparison on MNIST\n'
        'Row 1: RBM   Row 2: DBN   Row 3: VAE   '
        'Row 4: GAN   Row 5: DDPM',
        fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('figures/bonus_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/bonus_comparison.png')

Saved: figures/bonus_comparison.png


### 6.7 Analysis

| Model | Visual quality | Diversity | Stability | Key advantage | Key limitation |
|-------|---------------|-----------|-----------|---------------|----------------|
| **RBM** | Low -- noisy | Moderate | Very stable | Fast, simple | Single binary layer |
| **DBN** | Medium -- recognisable | Moderate | Stable | Hierarchical features | Approximate top-down pass |
| **VAE** | High -- smooth | Excellent | Very stable | Diverse, principled latent space | Slight blurring |
| **GAN** | High -- sharp edges | Variable | Unstable | Crisp pixel contrast | Mode collapse |
| **DDPM** | Very high -- realistic | Excellent | Stable | Best overall quality | Slow inference (T steps) |

**RBM:** The single hidden layer of 200 units provides a limited representation of MNIST.
Generated samples have recognisable digit outlines but noisy backgrounds. Gibbs sampling
with 300 steps helps, but the fundamental capacity limit remains.

**DBN:** Stacking two RBMs with greedy pre-training improves feature abstraction.
The second layer [200->100] captures coarser but more consistent structure,
producing cleaner overall shapes compared to the standalone RBM.

**VAE:** The encoder-decoder with latent z=10 and ELBO training produces the most diverse
samples. The per-batch normalisation to [0,1] inside the training loop is critical:
without it the binary cross-entropy reconstruction loss becomes poorly conditioned.

**GAN:** Using Tanh output (range [-1,1]) and training data also in [-1,1] is important
for stable adversarial training. The generated images are visually the sharpest among
the pixel-level models. Mode collapse is the main risk: running for 200 epochs with
Adam (lr=0.0002) provides a good balance between stability and quality.

**DDPM:** The full U-Net with sinusoidal time embedding, GroupNorm, SiLU, residual
connections and self-attention provides the richest noise predictor. Starting from
pure Gaussian noise and running T=1000 denoising steps produces sharp, diverse digits.
The main practical drawback is inference cost: each sample requires T=1000 forward
passes through the U-Net.

---
## 7. Conclusions <a id='sec7'></a>

### Summary of results

| Experiment | Best configuration | Best accuracy (pre-trained) | Gain over random |
|------------|-------------------|----------------------------|------------------|
| Depth (Fig. 1) | 3 layers x 200 | ~97.5-98% | ~1 pp |
| Width (Fig. 2) | 2 layers x 500 | ~98% | ~0.5 pp |
| Size n=1 000 (Fig. 3) | [784, 200, 200, 10] | ~90-92% | ~5-6 pp |
| Size n=60 000 (Fig. 3) | [784, 200, 200, 10] | ~97.5-98% | ~0.5 pp |
| Best overall | [784, 500, 500, 10] | ~98% | -- |

### Key conclusions

1. **Pre-training is decisive in two regimes.**
   - Few labels (n < 10 000): the unsupervised DBN phase exploits raw pixels
     before any label is seen, providing a 5-6 pp advantage.
   - Deep architectures (3+ layers): pre-training provides a warm start that avoids
     the vanishing gradient problem that degrades randomly-initialised deep networks.

2. **Optimal architecture.** `[784, 500, 500, 10]` pre-trained reaches ~98% accuracy.
   Adding a third layer or wider layers beyond 500 units yields negligible gains.

3. **Generative quality.** VAE and GAN surpass RBM and DBN in sample quality.
   However, for the purpose of initialising a classifier, the DBN is the most
   practical choice: fast, stable, and producing representations that transfer well.

4. **Confusion analysis.** The hardest digit pairs are 4/9, 3/5, and 7/1 --
   all cases where the binary pixel representations are genuinely ambiguous.

### Limitations and perspectives

- RBM and DBN use binary units; they do not extend naturally to continuous data.
- CD-1 is a biased gradient estimator; CD-k (k > 1) would improve accuracy.
- Modern regularisation techniques (BatchNorm, dropout, Adam) close the gap
  between pre-trained and random init on large datasets.
- Self-supervised pre-training (BERT, SimCLR, MAE) is the modern descendant
  of the DBN pre-training idea studied in this project.
